# Bloc 6 — Deep Learning for Sequential Energy Data
## Réseaux neuronaux pour la prévision de consommation électrique

## Étape 1 — Experimental Protocol & Strong Baselines
### Protocole expérimental et références de performance

### Partie A — Mise en pratique

L'objectif du Bloc 6 est de déterminer si les réseaux neuronaux apportent réellement une amélioration par rapport aux modèles déjà construits.

La tâche principale reste une prévision à :

$$
h = 30\text{ minutes}
$$

Les périodes sont séparées chronologiquement :

$$
Train < 2024
$$

$$
Validation = 2024
$$

$$
Evaluation\ du\ bloc = 2025
$$

$$
Test\ final = 2026
$$

Le jeu 2026 reste totalement intact.

Les hyperparamètres des réseaux seront choisis uniquement avec le train et la validation 2024.

In [127]:
import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)

# Dataset final enrichi météo
X = joblib.load(
    "data_processed/X_block3_weather_nonlinear.joblib"
)

y = joblib.load(
    "data_processed/y_block3_weather.joblib"
)

# Meilleur modèle de régression actuel
ridge_model = joblib.load(
    "models/block3_conformal_ridge.joblib"
)

train_mask = X.index < "2024-01-01"

val_mask = (
    (X.index >= "2024-01-01")
    & (X.index < "2025-01-01")
)

eval_mask = (
    (X.index >= "2025-01-01")
    & (X.index < "2026-01-01")
)

print("Train :", train_mask.sum())
print("Validation :", val_mask.sum())
print("Evaluation 2025 :", eval_mask.sum())
print("Features :", X.shape[1])

Train : 192482
Validation : 17568
Evaluation 2025 : 17517
Features : 429


In [128]:
y_2025 = y.loc[eval_mask].values

pred_day_naive = (
    X.loc[eval_mask, "lag_48"]
    .to_numpy()
)

pred_week_naive = (
    X.loc[eval_mask, "lag_336"]
    .to_numpy()
)

pred_ridge = ridge_model.predict(
    X.loc[eval_mask]
)

In [129]:
def regression_metrics(y_true, y_pred):

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    return {
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "MAPE_%": 100 * mean_absolute_percentage_error(
            y_true,
            y_pred
        ),
        "R2": r2_score(
            y_true,
            y_pred
        )
    }

In [130]:
baseline_results = []

for name, pred in [
    ("Naive 1 day", pred_day_naive),
    ("Naive 1 week", pred_week_naive),
    ("Ridge + Weather", pred_ridge)
]:

    metrics = regression_metrics(
        y_2025,
        pred
    )

    baseline_results.append({
        "Model": name,
        **metrics
    })

baseline_df = pd.DataFrame(
    baseline_results
)

print(
    baseline_df.round(4)
)

             Model           MSE       RMSE        MAE  MAPE_%      R2
0      Naive 1 day  1.326274e+07  3641.8050  2521.9668  5.0343  0.8824
1     Naive 1 week  2.650160e+07  5147.9702  3462.2436  6.4346  0.7650
2  Ridge + Weather  4.395791e+04   209.6614   151.7123  0.3074  0.9996


In [131]:
joblib.dump(
    baseline_df,
    "models/block6_baseline_metrics.joblib"
)

['models/block6_baseline_metrics.joblib']

In [132]:
import torch

print("PyTorch :", torch.__version__)
print("CUDA PyTorch :", torch.version.cuda)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print(
        "VRAM :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device :", device)

PyTorch : 2.14.0+cu130
CUDA PyTorch : 13.0
CUDA disponible : True
GPU : NVIDIA GeForce GTX 1650 Ti with Max-Q Design
VRAM : 4.0 GB
Device : cuda


## Étape 2 — MLP Baseline
### Réseau neuronal dense pour la prévision à 30 minutes

### Partie A — Mise en pratique

In [133]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import time

In [134]:
# -------------------------
# Données
# -------------------------

X_train_raw = X.loc[train_mask]
X_val_raw   = X.loc[val_mask]
X_eval_raw  = X.loc[eval_mask]

y_train_raw = y.loc[train_mask].to_numpy()
y_val_raw   = y.loc[val_mask].to_numpy()
y_eval_raw  = y.loc[eval_mask].to_numpy()

# -------------------------
# Standardisation X
# -------------------------

x_scaler = StandardScaler()

X_train = x_scaler.fit_transform(
    X_train_raw
).astype(np.float32)

X_val = x_scaler.transform(
    X_val_raw
).astype(np.float32)

X_eval = x_scaler.transform(
    X_eval_raw
).astype(np.float32)

# -------------------------
# Standardisation y
# -------------------------

y_scaler = StandardScaler()

y_train = y_scaler.fit_transform(
    y_train_raw.reshape(-1, 1)
).astype(np.float32).ravel()

y_val = y_scaler.transform(
    y_val_raw.reshape(-1, 1)
).astype(np.float32).ravel()

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Evaluation :", X_eval.shape)

Train : (192482, 429)
Validation : (17568, 429)
Evaluation : (17517, 429)


In [135]:
joblib.dump(
    x_scaler,
    "models/block6_mlp_x_scaler.joblib"
)

joblib.dump(
    y_scaler,
    "models/block6_mlp_y_scaler.joblib"
)

['models/block6_mlp_y_scaler.joblib']

In [136]:
BATCH_SIZE = 2048

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train[:, None])
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val[:, None])
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [137]:
class EnergyMLP(nn.Module):

    def __init__(
        self,
        n_features
    ):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                n_features,
                256
            ),

            nn.GELU(),

            nn.Dropout(0.10),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Dropout(0.10),

            nn.Linear(
                128,
                1
            )
        )

    def forward(self, x):
        return self.network(x)

In [138]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda")

model = EnergyMLP(
    X_train.shape[1]
).to(device)

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Paramètres :", n_params)
print("Device :", device)

Paramètres : 143105
Device : cuda


In [139]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

MAX_EPOCHS = 100
PATIENCE = 8

MAX_SECONDS = 300

use_amp = True

amp_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

best_val_loss = np.inf
patience_counter = 0

history = []

start_time = time.time()

MODEL_PATH = (
    "models/block6_mlp_direct.pt"
)

In [140]:
for epoch in range(1, MAX_EPOCHS + 1):

    # =====================
    # TRAIN
    # =====================

    model.train()

    train_loss_sum = 0
    train_n = 0

    timed_out = False

    for xb, yb in train_loader:

        if time.time() - start_time >= MAX_SECONDS:
            timed_out = True
            break

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):

            pred = model(xb)

            loss = criterion(
                pred,
                yb
            )

        amp_scaler.scale(
            loss
        ).backward()

        amp_scaler.step(
            optimizer
        )

        amp_scaler.update()

        train_loss_sum += (
            loss.item()
            * len(xb)
        )

        train_n += len(xb)

    if timed_out:
        print(
            "Limite de 5 minutes atteinte."
        )
        break

    train_loss = (
        train_loss_sum
        / train_n
    )

    # =====================
    # VALIDATION
    # =====================

    model.eval()

    val_loss_sum = 0
    val_n = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(
                device,
                non_blocking=True
            )

            yb = yb.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):

                pred = model(xb)

                loss = criterion(
                    pred,
                    yb
                )

            val_loss_sum += (
                loss.item()
                * len(xb)
            )

            val_n += len(xb)

    val_loss = (
        val_loss_sum
        / val_n
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    # sauvegarde progressive
    joblib.dump(
        history,
        "models/block6_mlp_history.joblib"
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Train={train_loss:.6f} | "
        f"Val={val_loss:.6f}"
    )

    # =====================
    # EARLY STOPPING
    # =====================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        patience_counter = 0

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print(
            "Early stopping."
        )

        break

Epoch 001 | Train=0.050397 | Val=0.003112
Epoch 002 | Train=0.006967 | Val=0.002037
Epoch 003 | Train=0.005208 | Val=0.001969
Epoch 004 | Train=0.004261 | Val=0.001227
Epoch 005 | Train=0.003792 | Val=0.001054
Epoch 006 | Train=0.003569 | Val=0.002670
Epoch 007 | Train=0.003323 | Val=0.001063
Epoch 008 | Train=0.003238 | Val=0.001753
Epoch 009 | Train=0.003191 | Val=0.000778
Epoch 010 | Train=0.002858 | Val=0.000919
Epoch 011 | Train=0.002920 | Val=0.000800
Epoch 012 | Train=0.003103 | Val=0.000626
Epoch 013 | Train=0.003039 | Val=0.000988
Epoch 014 | Train=0.002748 | Val=0.000634
Epoch 015 | Train=0.002727 | Val=0.000657
Epoch 016 | Train=0.002840 | Val=0.000687
Epoch 017 | Train=0.002887 | Val=0.000691
Epoch 018 | Train=0.002586 | Val=0.000800
Epoch 019 | Train=0.002724 | Val=0.001487
Epoch 020 | Train=0.002670 | Val=0.000567
Epoch 021 | Train=0.002597 | Val=0.000856
Limite de 5 minutes atteinte.


In [141]:
model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

model.eval()

EnergyMLP(
  (network): Sequential(
    (0): Linear(in_features=429, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): GELU(approximate='none')
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [142]:
X_eval_tensor = torch.from_numpy(
    X_eval
)

eval_loader = DataLoader(
    X_eval_tensor,
    batch_size=4096,
    shuffle=False
)

pred_scaled = []

with torch.no_grad():

    for xb in eval_loader:

        xb = xb.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = model(xb)

        pred_scaled.append(
            pred.float()
            .cpu()
            .numpy()
        )

pred_scaled = np.vstack(
    pred_scaled
)

pred_mlp = y_scaler.inverse_transform(
    pred_scaled
).ravel()

In [143]:
mlp_metrics = regression_metrics(
    y_eval_raw,
    pred_mlp
)

mlp_metrics

{'MSE': 83241.41646360702,
 'RMSE': np.float64(288.51588598135635),
 'MAE': 222.91575237626307,
 'MAPE_%': 0.4545104992643088,
 'R2': 0.9992618738635453}

In [144]:
comparison = pd.DataFrame([
    {
        "Model": "Ridge + Weather",
        **regression_metrics(
            y_eval_raw,
            pred_ridge
        )
    },

    {
        "Model": "Direct MLP",
        **mlp_metrics
    }
])

print(
    comparison.round(4)
)

             Model         MSE      RMSE       MAE  MAPE_%      R2
0  Ridge + Weather  43957.9064  209.6614  151.7123  0.3074  0.9996
1       Direct MLP  83241.4165  288.5159  222.9158  0.4545  0.9993


In [145]:
ridge_rmse = comparison.loc[
    comparison["Model"]
    == "Ridge + Weather",
    "RMSE"
].iloc[0]

mlp_rmse = comparison.loc[
    comparison["Model"]
    == "Direct MLP",
    "RMSE"
].iloc[0]

rmse_gain = (
    100
    * (
        ridge_rmse
        - mlp_rmse
    )
    / ridge_rmse
)

print(
    "RMSE gain vs Ridge :",
    rmse_gain,
    "%"
)

RMSE gain vs Ridge : -37.61039166904321 %


In [146]:
joblib.dump(
    {
        "metrics": comparison,
        "rmse_gain_pct": rmse_gain,
        "training_history": history,
        "n_parameters": n_params
    },
    "models/block6_mlp_direct_results.joblib"
)

['models/block6_mlp_direct_results.joblib']

### Partie C — Analyse

Le modèle Ridge suppose essentiellement une relation linéaire régularisée entre les variables et la consommation.

Le MLP introduit une fonction non linéaire :

$$
\hat y
=
f_\theta(X)
$$

Les couches cachées permettent notamment de modéliser des interactions complexes entre :

- consommation passée ;
- température ;
- heure ;
- jour ;
- saisonnalité ;
- variables dérivées.

Le modèle est néanmoins volontairement limité afin de déterminer si la non-linéarité apporte déjà une amélioration avant d'introduire des architectures temporelles plus complexes.

La référence à battre est :

$$
RMSE_{Ridge}=209.66\text{ MW}
$$

### Conclusion de l'Étape 2

Le MLP direct apprend correctement la relation entre les variables explicatives et la consommation, mais il ne dépasse pas le modèle Ridge.

Les performances 2025 sont :

| Modèle | RMSE | MAE | MAPE | R² |
|---|---:|---:|---:|---:|
| Ridge + Weather | 209.66 MW | 151.71 MW | 0.307 % | 0.9996 |
| Direct MLP | 288.52 MW | 222.92 MW | 0.455 % | 0.9993 |

Le RMSE du MLP est environ :

$$
37.6\%
$$

plus élevé que celui du Ridge.

Le modèle Ridge bénéficie déjà d'un feature engineering temporel et météorologique très riche, qui rend une grande partie du problème presque linéaire.

Le MLP direct doit au contraire apprendre l'ensemble de la fonction de prédiction avec plus de 143 000 paramètres.

Le MLP direct n'est donc pas retenu comme meilleur modèle.

La prochaine expérience consiste à utiliser le réseau uniquement pour apprendre les erreurs restantes du Ridge :

$$
\hat y_{final}
=
\hat y_{Ridge}
+
\hat\varepsilon_{MLP}
$$

In [147]:
from sklearn.base import clone
import os
import time

OOF_YEARS = [2022, 2023]

MAX_SECONDS = 300
start_time = time.time()

oof_parts = []

for year in OOF_YEARS:

    cache_path = (
        f"models/block6_ridge_oof_{year}.joblib"
    )

    # -----------------------------------
    # Recharge si déjà calculé
    # -----------------------------------

    if os.path.exists(cache_path):

        part = joblib.load(cache_path)

        print(
            year,
            ": chargé depuis le cache"
        )

    else:

        train_oof = (
            X.index
            < f"{year}-01-01"
        )

        test_oof = (
            (X.index >= f"{year}-01-01")
            &
            (X.index < f"{year + 1}-01-01")
        )

        print(
            f"{year} | "
            f"train={train_oof.sum()} | "
            f"prediction={test_oof.sum()}"
        )

        # Même Ridge,
        # mais ré-entraîné uniquement
        # sur le passé
        base_model = clone(
            ridge_model
        )

        base_model.fit(
            X.loc[train_oof],
            y.loc[train_oof]
        )

        ridge_pred = base_model.predict(
            X.loc[test_oof]
        )

        part = pd.DataFrame(
            {
                "y": (
                    y.loc[test_oof]
                    .to_numpy()
                ),

                "ridge_pred":
                    ridge_pred
            },
            index=X.loc[test_oof].index
        )

        part["residual"] = (
            part["y"]
            -
            part["ridge_pred"]
        )

        joblib.dump(
            part,
            cache_path
        )

        print(
            "Sauvegardé :",
            cache_path
        )

    oof_parts.append(part)

    print(
        "Temps écoulé :",
        round(time.time() - start_time, 1),
        "s"
    )

    if (
        time.time() - start_time
        >= MAX_SECONDS
    ):

        print(
            "Limite de 5 minutes atteinte."
        )

        break

2022 : chargé depuis le cache
Temps écoulé : 0.0 s
2023 : chargé depuis le cache
Temps écoulé : 0.0 s


In [148]:
residual_oof = pd.concat(
    oof_parts
).sort_index()

print(
    residual_oof.describe()
)

                  y    ridge_pred      residual
count  35040.000000  35040.000000  35040.000000
mean   50554.380422  50572.054918    -17.674496
std    10906.291028  10909.063821    243.616997
min    29798.000000  29673.158171  -3184.386418
25%    42493.750000  42500.309331   -155.611426
50%    48776.500000  48802.203006    -15.057644
75%    57378.500000  57376.131084    119.044841
max    86978.000000  86882.075682   2624.922463


In [149]:
joblib.dump(
    residual_oof,
    "models/block6_residual_oof.joblib"
)

['models/block6_residual_oof.joblib']

In [150]:
# =============================
# TRAIN : OOF 2022-2023
# =============================

res_train_index = (
    residual_oof.index
)

X_res_train_raw = X.loc[
    res_train_index
]

r_train_raw = (
    residual_oof["residual"]
    .to_numpy()
)


# =============================
# VALIDATION : 2024
# =============================

X_res_val_raw = X.loc[
    val_mask
]

y_res_val = (
    y.loc[val_mask]
    .to_numpy()
)

pred_ridge_val = ridge_model.predict(
    X_res_val_raw
)

r_val_raw = (
    y_res_val
    -
    pred_ridge_val
)


# =============================
# EVALUATION : 2025
# =============================

X_res_eval_raw = X.loc[
    eval_mask
]

y_res_eval = (
    y.loc[eval_mask]
    .to_numpy()
)

pred_ridge_eval = ridge_model.predict(
    X_res_eval_raw
)

r_eval_raw = (
    y_res_eval
    -
    pred_ridge_eval
)

print(
    "Residual train :",
    len(r_train_raw)
)

print(
    "Residual validation :",
    len(r_val_raw)
)

print(
    "Residual evaluation :",
    len(r_eval_raw)
)

print(
    "Train residual std :",
    r_train_raw.std()
)

print(
    "Validation residual std :",
    r_val_raw.std()
)

Residual train : 35040
Residual validation : 17568
Residual evaluation : 17517
Train residual std : 243.6135210243448
Validation residual std : 207.84024761268705


In [151]:
from sklearn.preprocessing import StandardScaler

res_x_scaler = StandardScaler()

X_res_train = (
    res_x_scaler
    .fit_transform(
        X_res_train_raw
    )
    .astype(np.float32)
)

X_res_val = (
    res_x_scaler
    .transform(
        X_res_val_raw
    )
    .astype(np.float32)
)

X_res_eval = (
    res_x_scaler
    .transform(
        X_res_eval_raw
    )
    .astype(np.float32)
)

In [152]:
res_y_scaler = StandardScaler()

r_train = (
    res_y_scaler
    .fit_transform(
        r_train_raw.reshape(-1, 1)
    )
    .astype(np.float32)
    .ravel()
)

r_val = (
    res_y_scaler
    .transform(
        r_val_raw.reshape(-1, 1)
    )
    .astype(np.float32)
    .ravel()
)

In [153]:
joblib.dump(
    res_x_scaler,
    "models/block6_residual_x_scaler.joblib"
)

joblib.dump(
    res_y_scaler,
    "models/block6_residual_y_scaler.joblib"
)

['models/block6_residual_y_scaler.joblib']

In [154]:
class ResidualMLP(nn.Module):

    def __init__(self, n_features):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                n_features,
                128
            ),

            nn.GELU(),

            nn.Dropout(0.05),

            nn.Linear(
                128,
                64
            ),

            nn.GELU(),

            nn.Dropout(0.05),

            nn.Linear(
                64,
                1
            )
        )

    def forward(self, x):

        return self.network(x)

In [155]:
torch.manual_seed(42)
np.random.seed(42)

res_model = ResidualMLP(
    X_res_train.shape[1]
).to(device)

res_n_params = sum(
    p.numel()
    for p in res_model.parameters()
    if p.requires_grad
)

print(
    "Residual MLP parameters :",
    res_n_params
)

Residual MLP parameters : 63361


In [156]:
res_train_dataset = TensorDataset(
    torch.from_numpy(
        X_res_train
    ),

    torch.from_numpy(
        r_train[:, None]
    )
)

res_train_loader = DataLoader(
    res_train_dataset,
    batch_size=2048,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

In [157]:
criterion = nn.SmoothL1Loss(
    beta=1.0
)

optimizer = torch.optim.AdamW(
    res_model.parameters(),
    lr=5e-4,
    weight_decay=5e-4
)

In [158]:
def predict_residual_scaled(
    model,
    X_array,
    batch_size=4096
):

    model.eval()

    preds = []

    loader = DataLoader(
        torch.from_numpy(
            X_array
        ),
        batch_size=batch_size,
        shuffle=False
    )

    with torch.no_grad():

        for xb in loader:

            xb = xb.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                pred = model(xb)

            preds.append(
                pred.float()
                .cpu()
                .numpy()
            )

    return np.vstack(preds)

In [159]:
MAX_EPOCHS = 100
PATIENCE = 10
MAX_SECONDS = 300

MODEL_PATH = (
    "models/block6_residual_mlp.pt"
)

best_val_rmse = np.inf
best_epoch = None

patience_counter = 0

history_res = []

amp_scaler = torch.amp.GradScaler(
    "cuda"
)

start_time = time.time()

In [160]:
for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # =========================
    # TRAIN
    # =========================

    res_model.train()

    running_loss = 0.0
    n_seen = 0

    timeout = False

    for xb, rb in res_train_loader:

        if (
            time.time() - start_time
            >= MAX_SECONDS
        ):

            timeout = True
            break

        xb = xb.to(
            device,
            non_blocking=True
        )

        rb = rb.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = res_model(xb)

            loss = criterion(
                pred,
                rb
            )

        amp_scaler.scale(
            loss
        ).backward()

        amp_scaler.step(
            optimizer
        )

        amp_scaler.update()

        running_loss += (
            loss.item()
            * len(xb)
        )

        n_seen += len(xb)

    if timeout:

        print(
            "Limite de 5 minutes atteinte."
        )

        break

    train_loss = (
        running_loss
        / n_seen
    )

    # =========================
    # VALIDATION 2024
    # =========================

    pred_r_val_scaled = (
        predict_residual_scaled(
            res_model,
            X_res_val
        )
    )

    pred_r_val = (
        res_y_scaler
        .inverse_transform(
            pred_r_val_scaled
        )
        .ravel()
    )

    # Correction du Ridge
    pred_final_val = (
        pred_ridge_val
        +
        pred_r_val
    )

    val_rmse = np.sqrt(
        mean_squared_error(
            y_res_val,
            pred_final_val
        )
    )

    history_res.append(
        {
            "epoch": epoch,
            "train_loss":
                train_loss,
            "val_rmse":
                val_rmse
        }
    )

    joblib.dump(
        history_res,
        "models/block6_residual_history.joblib"
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss={train_loss:.6f} | "
        f"Val RMSE={val_rmse:.3f}"
    )

    # =========================
    # BEST CHECKPOINT
    # =========================

    if val_rmse < best_val_rmse:

        best_val_rmse = val_rmse

        best_epoch = epoch

        patience_counter = 0

        torch.save(
            res_model.state_dict(),
            MODEL_PATH
        )

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print(
            "Early stopping."
        )

        break

Epoch 001 | Loss=0.381136 | Val RMSE=208.671
Epoch 002 | Loss=0.378340 | Val RMSE=207.993
Epoch 003 | Loss=0.376710 | Val RMSE=207.427
Epoch 004 | Loss=0.374536 | Val RMSE=206.801
Epoch 005 | Loss=0.372182 | Val RMSE=206.512
Epoch 006 | Loss=0.369451 | Val RMSE=204.140
Epoch 007 | Loss=0.365833 | Val RMSE=202.990
Epoch 008 | Loss=0.362049 | Val RMSE=202.093
Epoch 009 | Loss=0.360557 | Val RMSE=202.538
Epoch 010 | Loss=0.356104 | Val RMSE=199.818
Epoch 011 | Loss=0.352712 | Val RMSE=198.998
Epoch 012 | Loss=0.350429 | Val RMSE=198.240
Epoch 013 | Loss=0.348556 | Val RMSE=198.235
Epoch 014 | Loss=0.345949 | Val RMSE=196.393
Epoch 015 | Loss=0.342853 | Val RMSE=196.802
Epoch 016 | Loss=0.341441 | Val RMSE=195.734
Epoch 017 | Loss=0.339258 | Val RMSE=196.482
Epoch 018 | Loss=0.340245 | Val RMSE=195.359
Epoch 019 | Loss=0.336194 | Val RMSE=194.999
Epoch 020 | Loss=0.335595 | Val RMSE=194.562
Epoch 021 | Loss=0.333241 | Val RMSE=193.396
Epoch 022 | Loss=0.330819 | Val RMSE=194.083
Epoch 023 

In [161]:
res_model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

res_model.eval()

ResidualMLP(
  (network): Sequential(
    (0): Linear(in_features=429, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.05, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): GELU(approximate='none')
    (5): Dropout(p=0.05, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [162]:
pred_r_eval_scaled = (
    predict_residual_scaled(
        res_model,
        X_res_eval
    )
)

pred_r_eval = (
    res_y_scaler
    .inverse_transform(
        pred_r_eval_scaled
    )
    .ravel()
)

In [163]:
pred_residual_mlp = (
    pred_ridge_eval
    +
    pred_r_eval
)

In [164]:
ridge_metrics = regression_metrics(
    y_res_eval,
    pred_ridge_eval
)

residual_metrics = regression_metrics(
    y_res_eval,
    pred_residual_mlp
)

comparison_residual = pd.DataFrame([
    {
        "Model":
            "Ridge + Weather",
        **ridge_metrics
    },

    {
        "Model":
            "Ridge + Residual MLP",
        **residual_metrics
    }
])

print(
    comparison_residual.round(4)
)

                  Model         MSE      RMSE       MAE  MAPE_%      R2
0       Ridge + Weather  43957.9064  209.6614  151.7123  0.3074  0.9996
1  Ridge + Residual MLP  33204.0799  182.2199  132.2190  0.2670  0.9997


In [165]:
res_rmse_gain = (
    100
    * (
        ridge_metrics["RMSE"]
        -
        residual_metrics["RMSE"]
    )
    /
    ridge_metrics["RMSE"]
)

res_mae_gain = (
    100
    * (
        ridge_metrics["MAE"]
        -
        residual_metrics["MAE"]
    )
    /
    ridge_metrics["MAE"]
)

print(
    "RMSE gain :",
    res_rmse_gain,
    "%"
)

print(
    "MAE gain :",
    res_mae_gain,
    "%"
)

print(
    "Best epoch :",
    best_epoch
)

print(
    "Best validation RMSE :",
    best_val_rmse
)

RMSE gain : 13.088504039230811 %
MAE gain : 12.848893152103361 %
Best epoch : 99
Best validation RMSE : 180.33021623260356


In [166]:
residual_corr = np.corrcoef(
    r_eval_raw,
    pred_r_eval
)[0, 1]

print(
    "Correlation true/predicted residual :",
    residual_corr
)

print(
    "True residual std :",
    r_eval_raw.std()
)

print(
    "Predicted correction std :",
    pred_r_eval.std()
)

print(
    "Mean predicted correction :",
    pred_r_eval.mean()
)

Correlation true/predicted residual : 0.4980849884108042
True residual std : 209.66139114506808
Predicted correction std : 115.68106
Mean predicted correction : -4.9169283


In [167]:
joblib.dump(
    {
        "comparison":
            comparison_residual,

        "rmse_gain_pct":
            res_rmse_gain,

        "mae_gain_pct":
            res_mae_gain,

        "residual_corr":
            residual_corr,

        "best_epoch":
            best_epoch,

        "best_validation_rmse":
            best_val_rmse,

        "n_parameters":
            res_n_params,

        "history":
            history_res
    },
    "models/block6_residual_mlp_results.joblib"
)

['models/block6_residual_mlp_results.joblib']

### Partie C — Analyse

Le Ridge capture déjà l'essentiel de la consommation électrique.

Le Residual MLP ne cherche donc pas à reconstruire l'ensemble du signal :

$$
y_t
$$

mais uniquement la partie que le Ridge ne parvient pas à expliquer :

$$
r_t
=
y_t-\hat y_t^{Ridge}
$$

La prévision finale devient :

$$
\hat y_t^{final}
=
\hat y_t^{Ridge}
+
\hat r_t
$$

Les résidus utilisés pour l'apprentissage sont produits hors-échantillon avec un schéma temporel expansif afin d'éviter qu'un modèle de correction apprenne sur des erreurs artificiellement faibles obtenues en entraînement.

Une Huber loss est utilisée afin de réduire l'influence des erreurs extrêmes tout en conservant une fonction de perte différentiable.

Le meilleur réseau est sélectionné directement selon le RMSE final obtenu sur 2024.

## Étape 4 — Second-Order Residual Audit
### Analyse de la structure restante après la première correction

### Partie A — Mise en pratique

In [168]:
# Nouveau résidu après Ridge + Residual MLP

r2_eval = (
    y_res_eval
    - pred_residual_mlp
)

print(
    "Mean residual :",
    r2_eval.mean()
)

print(
    "Std residual :",
    r2_eval.std()
)

print(
    "RMSE :",
    np.sqrt(np.mean(r2_eval**2))
)

print(
    "MAE :",
    np.mean(np.abs(r2_eval))
)

Mean residual : 5.003240143112945
Std residual : 182.15116660241645
RMSE : 182.21986693703553
MAE : 132.2189649202074


In [169]:
lags = {
    "30 min": 1,
    "1 hour": 2,
    "1 day": 48,
    "1 week": 336
}

for name, lag in lags.items():

    corr = np.corrcoef(
        r2_eval[:-lag],
        r2_eval[lag:]
    )[0, 1]

    print(
        name,
        ":",
        corr
    )

30 min : 0.0735174780546058
1 hour : 0.024511182258297744
1 day : 0.03497331657845358
1 week : 0.09360788477198928


In [170]:
r2_df = pd.DataFrame(
    {
        "residual": r2_eval
    },
    index=X.loc[eval_mask].index
)

r2_df["hour"] = (
    r2_df.index.hour
)

r2_df["dow"] = (
    r2_df.index.dayofweek
)

r2_df["month"] = (
    r2_df.index.month
)

In [171]:
hour_bias = (
    r2_df
    .groupby("hour")["residual"]
    .mean()
)

dow_bias = (
    r2_df
    .groupby("dow")["residual"]
    .mean()
)

month_bias = (
    r2_df
    .groupby("month")["residual"]
    .mean()
)

print(
    "Max hourly bias :",
    hour_bias.abs().max()
)

print(
    "Max weekday bias :",
    dow_bias.abs().max()
)

print(
    "Max monthly bias :",
    month_bias.abs().max()
)

Max hourly bias : 30.451565165437394
Max weekday bias : 12.609921613526938
Max monthly bias : 17.274664110809947


In [172]:
corr_remaining_vs_correction = (
    np.corrcoef(
        r2_eval,
        pred_r_eval
    )[0, 1]
)

print(
    "Corr remaining residual / first correction :",
    corr_remaining_vs_correction
)

Corr remaining residual / first correction : -0.06177216252325333


In [173]:
second_residual_diagnostics = {
    "mean": r2_eval.mean(),
    "std": r2_eval.std(),
    "rmse": np.sqrt(
        np.mean(r2_eval**2)
    ),
    "mae": np.mean(
        np.abs(r2_eval)
    ),
    "hour_bias": hour_bias,
    "dow_bias": dow_bias,
    "month_bias": month_bias,
    "corr_first_correction":
        corr_remaining_vs_correction
}

joblib.dump(
    second_residual_diagnostics,
    "models/block6_second_residual_diagnostics.joblib"
)

['models/block6_second_residual_diagnostics.joblib']

### Partie C — Analyse

Le modèle construit jusqu'ici peut être écrit :

$$
f_1(X)
=
f_0(X)+g_1(X)
$$

où $f_0$ est le Ridge et $g_1$ le premier modèle de correction.

On définit maintenant un nouveau résidu :

$$
r_2
=
y-f_1(X)
$$

Si $r_2$ contient encore une structure prédictible, un second modèle peut être entraîné :

$$
g_2(X)\approx r_2
$$

et la prévision devient :

$$
f_2(X)
=
f_0(X)
+
g_1(X)
+
\eta g_2(X)
$$

Cette construction correspond à une approche additive proche du principe du gradient boosting.

Les corrections successives doivent cependant être validées hors-échantillon afin d'éviter d'apprendre progressivement le bruit.

## Étape 5 — Second-Stage Residual Learning
### Deuxième correction des erreurs restantes

### Partie A — Mise en pratique

In [174]:
res_model.load_state_dict(
    torch.load(
        "models/block6_residual_mlp.pt",
        map_location=device,
        weights_only=True
    )
)

res_model.eval()

pred_r_val_scaled = predict_residual_scaled(
    res_model,
    X_res_val
)

pred_r_val_best = (
    res_y_scaler
    .inverse_transform(pred_r_val_scaled)
    .ravel()
)

pred_stage1_2024 = (
    pred_ridge_val
    + pred_r_val_best
)

r2_2024 = (
    y_res_val
    - pred_stage1_2024
)

print("R2 residual 2024 mean :", r2_2024.mean())
print("R2 residual 2024 std  :", r2_2024.std())

R2 residual 2024 mean : 2.8871976495901137
R2 residual 2024 std  : 180.30710184634927


In [175]:
idx_2024 = X_res_val_raw.index

cut = pd.Timestamp(
    "2024-10-01",
    tz=idx_2024.tz
)

mask_g2_train = (
    idx_2024 < cut
)

mask_g2_val = (
    idx_2024 >= cut
)

X2_train_raw = X_res_val_raw.loc[
    mask_g2_train
]

X2_val_raw = X_res_val_raw.loc[
    mask_g2_val
]

r2_train_raw = r2_2024[
    mask_g2_train
]

r2_val_raw = r2_2024[
    mask_g2_val
]

print("Stage 2 train :", len(r2_train_raw))
print("Stage 2 val   :", len(r2_val_raw))

Stage 2 train : 13152
Stage 2 val   : 4416


In [176]:
g2_x_scaler = StandardScaler()

X2_train = g2_x_scaler.fit_transform(
    X2_train_raw
).astype(np.float32)

X2_val = g2_x_scaler.transform(
    X2_val_raw
).astype(np.float32)

X2_eval = g2_x_scaler.transform(
    X_res_eval_raw
).astype(np.float32)


g2_y_scaler = StandardScaler()

r2_train = (
    g2_y_scaler
    .fit_transform(
        r2_train_raw.reshape(-1, 1)
    )
    .astype(np.float32)
    .ravel()
)

r2_val = (
    g2_y_scaler
    .transform(
        r2_val_raw.reshape(-1, 1)
    )
    .astype(np.float32)
    .ravel()
)

In [177]:
joblib.dump(
    g2_x_scaler,
    "models/block6_stage2_x_scaler.joblib"
)

joblib.dump(
    g2_y_scaler,
    "models/block6_stage2_y_scaler.joblib"
)

['models/block6_stage2_y_scaler.joblib']

In [178]:
class ResidualMLP2(nn.Module):

    def __init__(self, n_features):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                n_features,
                64
            ),

            nn.GELU(),

            nn.Dropout(0.05),

            nn.Linear(
                64,
                32
            ),

            nn.GELU(),

            nn.Dropout(0.05),

            nn.Linear(
                32,
                1
            )
        )

    def forward(self, x):
        return self.network(x)

In [179]:
torch.manual_seed(42)

model2 = ResidualMLP2(
    X2_train.shape[1]
).to(device)

n_params2 = sum(
    p.numel()
    for p in model2.parameters()
    if p.requires_grad
)

print(
    "Stage 2 parameters :",
    n_params2
)

Stage 2 parameters : 29633


In [180]:
train2_dataset = TensorDataset(
    torch.from_numpy(X2_train),
    torch.from_numpy(r2_train[:, None])
)

train2_loader = DataLoader(
    train2_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

criterion2 = nn.SmoothL1Loss(
    beta=1.0
)

optimizer2 = torch.optim.AdamW(
    model2.parameters(),
    lr=3e-4,
    weight_decay=1e-3
)

scaler2 = torch.amp.GradScaler(
    "cuda"
)

MAX_SECONDS = 300
MAX_EPOCHS = 150
PATIENCE = 12

best_rmse2 = np.inf
best_epoch2 = None
patience = 0

history2 = []

start = time.time()

PATH2 = (
    "models/block6_residual_mlp_stage2.pt"
)

In [181]:
for epoch in range(1, MAX_EPOCHS + 1):

    model2.train()

    loss_sum = 0
    n_seen = 0
    timeout = False

    for xb, yb in train2_loader:

        if time.time() - start >= MAX_SECONDS:
            timeout = True
            break

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        optimizer2.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = model2(xb)

            loss = criterion2(
                pred,
                yb
            )

        scaler2.scale(
            loss
        ).backward()

        scaler2.step(
            optimizer2
        )

        scaler2.update()

        loss_sum += (
            loss.item()
            * len(xb)
        )

        n_seen += len(xb)

    if timeout:
        print(
            "Limite de 5 minutes atteinte."
        )
        break

    train_loss = (
        loss_sum / n_seen
    )

    # Validation
    pred2_scaled = predict_residual_scaled(
        model2,
        X2_val
    )

    pred2_val = (
        g2_y_scaler
        .inverse_transform(
            pred2_scaled
        )
        .ravel()
    )

    residual_rmse = np.sqrt(
        mean_squared_error(
            r2_val_raw,
            pred2_val
        )
    )

    history2.append({
        "epoch": epoch,
        "loss": train_loss,
        "val_residual_rmse":
            residual_rmse
    })

    print(
        f"Epoch {epoch:03d} | "
        f"Loss={train_loss:.5f} | "
        f"Residual RMSE={residual_rmse:.3f}"
    )

    if residual_rmse < best_rmse2:

        best_rmse2 = residual_rmse
        best_epoch2 = epoch
        patience = 0

        torch.save(
            model2.state_dict(),
            PATH2
        )

    else:

        patience += 1

    if patience >= PATIENCE:

        print("Early stopping.")
        break

Epoch 001 | Loss=0.37332 | Residual RMSE=205.476
Epoch 002 | Loss=0.37020 | Residual RMSE=205.654
Epoch 003 | Loss=0.36818 | Residual RMSE=205.646
Epoch 004 | Loss=0.36627 | Residual RMSE=205.726
Epoch 005 | Loss=0.36477 | Residual RMSE=205.822
Epoch 006 | Loss=0.36319 | Residual RMSE=205.627
Epoch 007 | Loss=0.36146 | Residual RMSE=205.775
Epoch 008 | Loss=0.36031 | Residual RMSE=205.614
Epoch 009 | Loss=0.35941 | Residual RMSE=205.505
Epoch 010 | Loss=0.35819 | Residual RMSE=205.494
Epoch 011 | Loss=0.35787 | Residual RMSE=205.102
Epoch 012 | Loss=0.35581 | Residual RMSE=205.485
Epoch 013 | Loss=0.35493 | Residual RMSE=205.218
Epoch 014 | Loss=0.35388 | Residual RMSE=205.432
Epoch 015 | Loss=0.35321 | Residual RMSE=205.293
Epoch 016 | Loss=0.35232 | Residual RMSE=205.521
Epoch 017 | Loss=0.35163 | Residual RMSE=205.949
Epoch 018 | Loss=0.35129 | Residual RMSE=205.633
Epoch 019 | Loss=0.34957 | Residual RMSE=205.061
Epoch 020 | Loss=0.34911 | Residual RMSE=205.765
Epoch 021 | Loss=0.3

In [182]:
model2.load_state_dict(
    torch.load(
        PATH2,
        map_location=device,
        weights_only=True
    )
)

<All keys matched successfully>

In [183]:
pred2_val_scaled = predict_residual_scaled(
    model2,
    X2_val
)

pred2_val = (
    g2_y_scaler
    .inverse_transform(
        pred2_val_scaled
    )
    .ravel()
)

In [184]:
etas = [
    0.0,
    0.25,
    0.50,
    0.75,
    1.0
]

base_val = pred_stage1_2024[
    mask_g2_val
]

y_q4 = y_res_val[
    mask_g2_val
]

eta_results = []

for eta in etas:

    pred = (
        base_val
        + eta * pred2_val
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_q4,
            pred
        )
    )

    eta_results.append(
        [eta, rmse]
    )

eta_df = pd.DataFrame(
    eta_results,
    columns=[
        "eta",
        "RMSE"
    ]
)

print(eta_df)

best_eta = eta_df.loc[
    eta_df["RMSE"].idxmin(),
    "eta"
]

print(
    "Best eta :",
    best_eta
)

    eta        RMSE
0  0.00  205.531274
1  0.25  204.866344
2  0.50  204.565508
3  0.75  204.630372
4  1.00  205.060590
Best eta : 0.5


In [185]:
pred2_eval_scaled = predict_residual_scaled(
    model2,
    X2_eval
)

pred2_eval = (
    g2_y_scaler
    .inverse_transform(
        pred2_eval_scaled
    )
    .ravel()
)

In [186]:
pred_stage2_2025 = (
    pred_residual_mlp
    +
    best_eta * pred2_eval
)

In [187]:
stage2_metrics = regression_metrics(
    y_res_eval,
    pred_stage2_2025
)

comparison_stage2 = pd.DataFrame([
    {
        "Model":
            "Ridge",

        **ridge_metrics
    },

    {
        "Model":
            "Ridge + MLP1",

        **residual_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + MLP2",

        **stage2_metrics
    }
])

print(
    comparison_stage2.round(4)
)

                 Model         MSE      RMSE       MAE  MAPE_%      R2
0                Ridge  43957.9064  209.6614  151.7123  0.3074  0.9996
1         Ridge + MLP1  33204.0799  182.2199  132.2190  0.2670  0.9997
2  Ridge + MLP1 + MLP2  32442.3019  180.1175  130.2853  0.2629  0.9997


In [188]:
gain_stage2 = (
    100
    * (
        residual_metrics["RMSE"]
        -
        stage2_metrics["RMSE"]
    )
    /
    residual_metrics["RMSE"]
)

print(
    "Additional RMSE gain :",
    gain_stage2,
    "%"
)

Additional RMSE gain : 1.1537710581118905 %


In [189]:
corr2 = np.corrcoef(
    r2_eval,
    pred2_eval
)[0, 1]

r3_eval = (
    y_res_eval
    -
    pred_stage2_2025
)

print(
    "Correlation r2 / predicted correction :",
    corr2
)

print(
    "New residual std :",
    r3_eval.std()
)

print(
    "New RMSE :",
    np.sqrt(
        np.mean(r3_eval**2)
    )
)

Correlation r2 / predicted correction : 0.16339640515654058
New residual std : 180.09734092850294
New RMSE : 180.11746685018602


In [190]:
joblib.dump(
    {
        "comparison":
            comparison_stage2,

        "best_eta":
            best_eta,

        "best_epoch":
            best_epoch2,

        "validation":
            eta_df,

        "additional_gain_pct":
            gain_stage2,

        "residual_correlation":
            corr2,

        "n_parameters":
            n_params2
    },
    "models/block6_stage2_results.joblib"
)

['models/block6_stage2_results.joblib']

### Partie C — Analyse

Après la première correction, les résidus sont beaucoup moins structurés.

Leur moyenne est proche de zéro et leurs autocorrélations sont faibles, bien qu'une petite dépendance hebdomadaire et à très court terme subsiste.

Un deuxième modèle de correction est donc testé :

$$
f_2(X)
=
f_1(X)
+
\eta g_2(X)
$$

Le coefficient $\eta$ contrôle l'intensité de la nouvelle correction.

Si :

$$
\eta=0
$$

est sélectionné, la deuxième correction n'apporte pas suffisamment d'information et le processus doit être arrêté.

Si :

$$
\eta>0
$$

et que le RMSE 2025 diminue, cela montre qu'une deuxième structure résiduelle était encore exploitable.

Les corrections successives ne doivent pas être poursuivies automatiquement : le processus est arrêté dès qu'une nouvelle étape ne produit plus de gain hors-échantillon.

## Étape 6 — Temporal Residual Learning
### Modélisation temporelle des erreurs restantes

### Partie A — Mise en pratique

In [191]:
r2_2024_series = pd.Series(
    r2_2024,
    index=X_res_val_raw.index,
    name="r2"
)

r2_2025_series = pd.Series(
    r2_eval,
    index=X_res_eval_raw.index,
    name="r2"
)

r2_all = pd.concat([
    r2_2024_series,
    r2_2025_series
]).sort_index()

print(r2_all.head())
print(r2_all.tail())

Date et Heure
2024-01-01 00:00:00+00:00   -149.966481
2024-01-01 00:30:00+00:00   -365.947107
2024-01-01 01:00:00+00:00    117.529331
2024-01-01 01:30:00+00:00    -19.435616
2024-01-01 02:00:00+00:00    -81.650054
Freq: 30min, Name: r2, dtype: float64
Date et Heure
2025-12-31 20:00:00+00:00    183.885093
2025-12-31 20:30:00+00:00     36.787025
2025-12-31 21:00:00+00:00    328.407219
2025-12-31 21:30:00+00:00    450.481921
2025-12-31 22:00:00+00:00    172.640765
Freq: 30min, Name: r2, dtype: float64


In [192]:
res_ar = pd.DataFrame(
    index=r2_all.index
)

res_ar["target"] = r2_all

for lag in [1, 2, 48, 336]:
    res_ar[f"lag_{lag}"] = (
        r2_all.shift(lag)
    )

res_ar = res_ar.dropna()

In [193]:
cut = pd.Timestamp(
    "2024-10-01",
    tz=res_ar.index.tz
)

mask_ar_train = (
    res_ar.index < cut
)

mask_ar_val = (
    (res_ar.index >= cut)
    &
    (res_ar.index < "2025-01-01")
)

mask_ar_eval = (
    (res_ar.index >= "2025-01-01")
    &
    (res_ar.index < "2026-01-01")
)

In [194]:
from sklearn.linear_model import Ridge

features_ar = [
    "lag_1",
    "lag_2",
    "lag_48",
    "lag_336"
]

ar_model = Ridge(
    alpha=1.0
)

ar_model.fit(
    res_ar.loc[
        mask_ar_train,
        features_ar
    ],
    res_ar.loc[
        mask_ar_train,
        "target"
    ]
)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [195]:
pred_ar_val = ar_model.predict(
    res_ar.loc[
        mask_ar_val,
        features_ar
    ]
)

rmse_ar_val = np.sqrt(
    mean_squared_error(
        res_ar.loc[
            mask_ar_val,
            "target"
        ],
        pred_ar_val
    )
)

print(
    "AR residual validation RMSE :",
    rmse_ar_val
)

AR residual validation RMSE : 203.7126899517026


In [196]:
pred_ar_2025 = ar_model.predict(
    res_ar.loc[
        mask_ar_eval,
        features_ar
    ]
)

In [197]:
LOOKBACK = 336

def make_sequences(
    series,
    lookback
):

    values = (
        series.to_numpy()
        .astype(np.float32)
    )

    X_seq = []
    y_seq = []
    idx_seq = []

    for i in range(
        lookback,
        len(values)
    ):

        X_seq.append(
            values[
                i-lookback:i
            ]
        )

        y_seq.append(
            values[i]
        )

        idx_seq.append(
            series.index[i]
        )

    X_seq = np.array(
        X_seq,
        dtype=np.float32
    )

    y_seq = np.array(
        y_seq,
        dtype=np.float32
    )

    idx_seq = pd.Index(
        idx_seq
    )

    return (
        X_seq,
        y_seq,
        idx_seq
    )

In [198]:
X_seq, y_seq, seq_index = (
    make_sequences(
        r2_all,
        LOOKBACK
    )
)

print(
    "Sequences :",
    X_seq.shape
)

Sequences : (34749, 336)


In [199]:
X_seq = X_seq[:, :, None]

print(
    X_seq.shape
)

(34749, 336, 1)


In [200]:
cut = pd.Timestamp(
    "2024-10-01",
    tz=seq_index.tz
)

train_seq_mask = (
    seq_index < cut
)

val_seq_mask = (
    (seq_index >= cut)
    &
    (seq_index < "2025-01-01")
)

eval_seq_mask = (
    (seq_index >= "2025-01-01")
    &
    (seq_index < "2026-01-01")
)

X_seq_train = X_seq[
    train_seq_mask
]

y_seq_train = y_seq[
    train_seq_mask
]

X_seq_val = X_seq[
    val_seq_mask
]

y_seq_val = y_seq[
    val_seq_mask
]

X_seq_eval = X_seq[
    eval_seq_mask
]

y_seq_eval = y_seq[
    eval_seq_mask
]

eval_seq_index = seq_index[
    eval_seq_mask
]

print(
    "Train :", X_seq_train.shape
)

print(
    "Val   :", X_seq_val.shape
)

print(
    "Eval  :", X_seq_eval.shape
)

Train : (12816, 336, 1)
Val   : (4416, 336, 1)
Eval  : (17517, 336, 1)


In [201]:
seq_mean = (
    X_seq_train.mean()
)

seq_std = (
    X_seq_train.std()
)

X_seq_train = (
    (X_seq_train - seq_mean)
    / seq_std
)

X_seq_val = (
    (X_seq_val - seq_mean)
    / seq_std
)

X_seq_eval = (
    (X_seq_eval - seq_mean)
    / seq_std
)

y_seq_train_scaled = (
    (y_seq_train - seq_mean)
    / seq_std
)

y_seq_val_scaled = (
    (y_seq_val - seq_mean)
    / seq_std
)

In [202]:
joblib.dump(
    {
        "mean": seq_mean,
        "std": seq_std
    },
    "models/block6_gru_residual_scaler.joblib"
)

['models/block6_gru_residual_scaler.joblib']

In [203]:
class ResidualGRU(nn.Module):

    def __init__(
        self,
        hidden_size=32
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        output, hidden = self.gru(x)

        last_hidden = hidden[-1]

        return self.fc(
            last_hidden
        )

In [204]:
train_gru_dataset = TensorDataset(
    torch.from_numpy(
        X_seq_train
    ),
    torch.from_numpy(
        y_seq_train_scaled[:, None]
    )
)

val_gru_dataset = TensorDataset(
    torch.from_numpy(
        X_seq_val
    ),
    torch.from_numpy(
        y_seq_val_scaled[:, None]
    )
)

train_gru_loader = DataLoader(
    train_gru_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_gru_loader = DataLoader(
    val_gru_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [205]:
torch.manual_seed(42)

gru_model = ResidualGRU(
    hidden_size=32
).to(device)

optimizer_gru = torch.optim.AdamW(
    gru_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

criterion_gru = nn.SmoothL1Loss()

scaler_gru = torch.amp.GradScaler(
    "cuda"
)

MAX_SECONDS = 300
MAX_EPOCHS = 100
PATIENCE = 10

best_gru_rmse = np.inf
best_gru_epoch = None

patience_gru = 0

history_gru = []

PATH_GRU = (
    "models/block6_residual_gru.pt"
)

start = time.time()

In [206]:
for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    gru_model.train()

    loss_sum = 0
    n_seen = 0

    timeout = False

    for xb, yb in train_gru_loader:

        if (
            time.time() - start
            >= MAX_SECONDS
        ):

            timeout = True
            break

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        optimizer_gru.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = gru_model(xb)

            loss = criterion_gru(
                pred,
                yb
            )

        scaler_gru.scale(
            loss
        ).backward()

        scaler_gru.step(
            optimizer_gru
        )

        scaler_gru.update()

        loss_sum += (
            loss.item()
            * len(xb)
        )

        n_seen += len(xb)

    if timeout:

        print(
            "Limite de 5 minutes atteinte."
        )
        break

    train_loss = (
        loss_sum / n_seen
    )

    # Validation
    gru_model.eval()

    preds = []

    with torch.no_grad():

        for xb, _ in val_gru_loader:

            xb = xb.to(device)

            pred = gru_model(xb)

            preds.append(
                pred.cpu().numpy()
            )

    pred_val_scaled = np.vstack(
        preds
    ).ravel()

    pred_val = (
        pred_val_scaled
        * seq_std
        + seq_mean
    )

    rmse_val = np.sqrt(
        mean_squared_error(
            y_seq_val,
            pred_val
        )
    )

    history_gru.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_rmse": rmse_val
    })

    joblib.dump(
        history_gru,
        "models/block6_residual_gru_history.joblib"
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss={train_loss:.5f} | "
        f"Val RMSE={rmse_val:.3f}"
    )

    if rmse_val < best_gru_rmse:

        best_gru_rmse = rmse_val

        best_gru_epoch = epoch

        patience_gru = 0

        torch.save(
            gru_model.state_dict(),
            PATH_GRU
        )

    else:

        patience_gru += 1

    if patience_gru >= PATIENCE:

        print(
            "Early stopping."
        )
        break

Epoch 001 | Loss=0.36961 | Val RMSE=204.942
Epoch 002 | Loss=0.36868 | Val RMSE=205.132
Epoch 003 | Loss=0.36796 | Val RMSE=205.552
Epoch 004 | Loss=0.36783 | Val RMSE=205.740
Epoch 005 | Loss=0.36757 | Val RMSE=205.842
Epoch 006 | Loss=0.36753 | Val RMSE=206.086
Epoch 007 | Loss=0.36762 | Val RMSE=206.107
Epoch 008 | Loss=0.36756 | Val RMSE=206.186
Epoch 009 | Loss=0.36786 | Val RMSE=205.973
Epoch 010 | Loss=0.36751 | Val RMSE=206.086
Epoch 011 | Loss=0.36754 | Val RMSE=205.972
Early stopping.


In [207]:
gru_model.load_state_dict(
    torch.load(
        PATH_GRU,
        map_location=device,
        weights_only=True
    )
)

gru_model.eval()

ResidualGRU(
  (gru): GRU(1, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

In [208]:
eval_gru_loader = DataLoader(
    torch.from_numpy(
        X_seq_eval
    ),
    batch_size=512,
    shuffle=False
)

pred_gru = []

with torch.no_grad():

    for xb in eval_gru_loader:

        xb = xb.to(device)

        pred = gru_model(xb)

        pred_gru.append(
            pred.cpu().numpy()
        )

pred_gru = np.vstack(
    pred_gru
).ravel()

pred_gru = (
    pred_gru
    * seq_std
    + seq_mean
)

In [209]:
base_stage1_series = pd.Series(
    pred_residual_mlp,
    index=X_res_eval_raw.index
)

base_stage1_gru = (
    base_stage1_series.loc[
        eval_seq_index
    ]
    .to_numpy()
)

y_eval_gru = (
    y.loc[
        eval_seq_index
    ]
    .to_numpy()
)

In [210]:
pred_stage1_gru = (
    base_stage1_gru
    + pred_gru
)

gru_metrics = regression_metrics(
    y_eval_gru,
    pred_stage1_gru
)

In [211]:
print(
    "GRU residual metrics :",
    gru_metrics
)

GRU residual metrics : {'MSE': 33055.62202990366, 'RMSE': np.float64(181.81205138797498), 'MAE': 131.867943441588, 'MAPE_%': 0.26646709472329083, 'R2': 0.9997068860716983}


### Conclusion — Temporal Residual Learning

Après la première correction par MLP, les résidus conservent une faible structure temporelle.

Le modèle autorégressif obtient sur la validation :

$$
RMSE_{AR}=203.71\text{ MW}
$$

Le GRU améliore légèrement le modèle Ridge + MLP1 sur 2025 :

$$
182.22
\rightarrow
181.81\text{ MW}
$$

soit environ :

$$
0.22\%
$$

Cependant, il reste moins performant que la deuxième correction MLP :

$$
RMSE_{MLP2}=180.12\text{ MW}
$$

La meilleure performance du GRU est obtenue dès la première epoch, puis la validation se détériore.

Cela suggère que la dépendance temporelle résiduelle existe mais qu'elle est faible et facilement sur-apprise.

Le modèle Ridge + MLP1 + MLP2 reste donc la meilleure architecture à ce stade.

In [212]:
# RMSE sans correction AR
baseline_ar_val_rmse = np.sqrt(
    mean_squared_error(
        res_ar.loc[
            mask_ar_val,
            "target"
        ],
        np.zeros(mask_ar_val.sum())
    )
)

etas_ar = [
    0.0,
    0.25,
    0.50,
    0.75,
    1.0
]

ar_eta_results = []

true_ar_val = res_ar.loc[
    mask_ar_val,
    "target"
].to_numpy()

for eta in etas_ar:

    corrected_residual = (
        eta * pred_ar_val
    )

    rmse = np.sqrt(
        mean_squared_error(
            true_ar_val,
            corrected_residual
        )
    )

    ar_eta_results.append(
        [eta, rmse]
    )

ar_eta_df = pd.DataFrame(
    ar_eta_results,
    columns=[
        "eta",
        "Residual_RMSE"
    ]
)

print(ar_eta_df)

best_eta_ar = ar_eta_df.loc[
    ar_eta_df["Residual_RMSE"].idxmin(),
    "eta"
]

print(
    "Best AR eta :",
    best_eta_ar
)

    eta  Residual_RMSE
0  0.00     205.531274
1  0.25     204.845940
2  0.50     204.313656
3  0.75     203.935620
4  1.00     203.712690
Best AR eta : 1.0


In [213]:
ar_eval_index = (
    res_ar.loc[
        mask_ar_eval
    ].index
)

base_stage1_ar = (
    pd.Series(
        pred_residual_mlp,
        index=X_res_eval_raw.index
    )
    .loc[ar_eval_index]
    .to_numpy()
)

y_ar_eval = (
    y.loc[
        ar_eval_index
    ]
    .to_numpy()
)

pred_ar_final = (
    base_stage1_ar
    +
    best_eta_ar
    * pred_ar_2025
)

ar_metrics = regression_metrics(
    y_ar_eval,
    pred_ar_final
)

print(
    "AR corrected metrics :",
    ar_metrics
)

AR corrected metrics : {'MSE': 32673.98173928273, 'RMSE': np.float64(180.7594582291138), 'MAE': 131.20606817155053, 'MAPE_%': 0.2650648452272172, 'R2': 0.9997102701884661}


In [214]:
temporal_comparison = pd.DataFrame([
    {
        "Model":
            "Ridge + MLP1",

        **regression_metrics(
            y_ar_eval,
            base_stage1_ar
        )
    },

    {
        "Model":
            "Ridge + MLP1 + AR",

        **ar_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + GRU",

        **gru_metrics
    }
])

print(
    temporal_comparison.round(4)
)

                Model         MSE      RMSE       MAE  MAPE_%      R2
0        Ridge + MLP1  33204.0799  182.2199  132.2190  0.2670  0.9997
1   Ridge + MLP1 + AR  32673.9817  180.7595  131.2061  0.2651  0.9997
2  Ridge + MLP1 + GRU  33055.6220  181.8121  131.8679  0.2665  0.9997


In [215]:
joblib.dump(
    {
        "eta_table":
            ar_eta_df,

        "best_eta":
            best_eta_ar,

        "metrics":
            temporal_comparison
    },
    "models/block6_ar_residual_results.joblib"
)

['models/block6_ar_residual_results.joblib']

### Conclusion — Modélisation autorégressive des résidus

La structure temporelle restante après la première correction est faible mais exploitable.

Le modèle autorégressif améliore le RMSE :

$$
182.22
\rightarrow
180.76\text{ MW}
$$

Il est plus performant que le GRU :

$$
181.81\text{ MW}
$$

malgré une complexité beaucoup plus faible.

Le deuxième MLP reste néanmoins légèrement meilleur :

$$
RMSE=180.12\text{ MW}
$$

Ces résultats montrent qu'une petite structure temporelle persiste dans les résidus, mais qu'un modèle complexe n'est pas nécessairement le plus efficace pour l'exploiter.

## Étape 7 — Temporal Convolutional Network
### Correction convolutionnelle des résidus temporels

### Partie A — Mise en pratique

In [216]:
DILATIONS = [
    1, 2, 4, 8,
    16, 32, 64, 128
]

In [217]:
class CausalTCNBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        dilation,
        kernel_size=3,
        dropout=0.05
    ):

        super().__init__()

        padding = (
            (kernel_size - 1)
            * dilation
        )

        self.net = nn.Sequential(

            # Padding uniquement dans le passé
            nn.ConstantPad1d(
                (padding, 0),
                0
            ),

            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                dilation=dilation
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.ConstantPad1d(
                (padding, 0),
                0
            ),

            nn.Conv1d(
                out_channels,
                out_channels,
                kernel_size=kernel_size,
                dilation=dilation
            ),

            nn.GELU(),

            nn.Dropout(dropout)
        )

        # Pour rendre possible :
        # sortie + entrée
        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        )

    def forward(self, x):

        return (
            self.net(x)
            +
            self.residual(x)
        )

In [218]:
class ResidualTCN(nn.Module):

    def __init__(
        self,
        channels=16
    ):

        super().__init__()

        layers = []

        in_ch = 1

        for dilation in DILATIONS:

            layers.append(
                CausalTCNBlock(
                    in_channels=in_ch,
                    out_channels=channels,
                    dilation=dilation
                )
            )

            in_ch = channels

        self.tcn = nn.Sequential(
            *layers
        )

        self.output = nn.Linear(
            channels,
            1
        )

    def forward(self, x):

        # DataLoader :
        # batch x time x features

        # Conv1d veut :
        # batch x features x time
        x = x.transpose(
            1,
            2
        )

        x = self.tcn(x)

        # représentation du dernier instant
        x = x[:, :, -1]

        return self.output(x)

In [219]:
torch.manual_seed(42)

tcn_model = ResidualTCN(
    channels=16
).to(device)

tcn_params = sum(
    p.numel()
    for p in tcn_model.parameters()
    if p.requires_grad
)

print(
    "TCN parameters :",
    tcn_params
)

TCN parameters : 11873


In [220]:
tcn_train_dataset = TensorDataset(
    torch.from_numpy(
        X_seq_train
    ),
    torch.from_numpy(
        y_seq_train_scaled[:, None]
    )
)

tcn_train_loader = DataLoader(
    tcn_train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

In [221]:
criterion_tcn = nn.SmoothL1Loss()

optimizer_tcn = torch.optim.AdamW(
    tcn_model.parameters(),
    lr=3e-4,
    weight_decay=5e-4
)

scaler_tcn = torch.amp.GradScaler(
    "cuda"
)

MAX_SECONDS = 300
MAX_EPOCHS = 100
PATIENCE = 10

best_tcn_rmse = np.inf
best_tcn_epoch = None

patience_tcn = 0
history_tcn = []

TCN_PATH = (
    "models/block6_residual_tcn.pt"
)

start = time.time()

In [222]:
for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    tcn_model.train()

    loss_sum = 0.0
    n_seen = 0
    timeout = False

    for xb, yb in tcn_train_loader:

        if time.time() - start >= MAX_SECONDS:

            timeout = True
            break

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        optimizer_tcn.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = tcn_model(xb)

            loss = criterion_tcn(
                pred,
                yb
            )

        scaler_tcn.scale(
            loss
        ).backward()

        scaler_tcn.step(
            optimizer_tcn
        )

        scaler_tcn.update()

        loss_sum += (
            loss.item()
            * len(xb)
        )

        n_seen += len(xb)

    if timeout:

        print(
            "Limite de 5 minutes atteinte."
        )
        break

    train_loss = (
        loss_sum / n_seen
    )

    tcn_model.eval()

    pred_val_tcn = []

    with torch.no_grad():

        for xb, _ in val_gru_loader:

            xb = xb.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                pred = tcn_model(xb)

            pred_val_tcn.append(
                pred.float()
                .cpu()
                .numpy()
            )

    pred_val_tcn = (
        np.vstack(
            pred_val_tcn
        ).ravel()
    )

    pred_val_tcn = (
        pred_val_tcn
        * seq_std
        + seq_mean
    )

    val_tcn_rmse = np.sqrt(
        mean_squared_error(
            y_seq_val,
            pred_val_tcn
        )
    )

    history_tcn.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_rmse": val_tcn_rmse
    })

    joblib.dump(
        history_tcn,
        "models/block6_tcn_history.joblib"
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss={train_loss:.5f} | "
        f"Val RMSE={val_tcn_rmse:.3f}"
    )

    if val_tcn_rmse < best_tcn_rmse:

        best_tcn_rmse = (
            val_tcn_rmse
        )

        best_tcn_epoch = epoch

        patience_tcn = 0

        torch.save(
            tcn_model.state_dict(),
            TCN_PATH
        )

    else:

        patience_tcn += 1

    if patience_tcn >= PATIENCE:

        print(
            "Early stopping."
        )
        break

        tcn_model.eval()

    pred_val_tcn = []

    with torch.no_grad():

        for xb, _ in val_gru_loader:

            xb = xb.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                pred = tcn_model(xb)

            pred_val_tcn.append(
                pred.float()
                .cpu()
                .numpy()
            )

    pred_val_tcn = (
        np.vstack(
            pred_val_tcn
        ).ravel()
    )

    pred_val_tcn = (
        pred_val_tcn
        * seq_std
        + seq_mean
    )

    val_tcn_rmse = np.sqrt(
        mean_squared_error(
            y_seq_val,
            pred_val_tcn
        )
    )

    history_tcn.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_rmse": val_tcn_rmse
    })

    joblib.dump(
        history_tcn,
        "models/block6_tcn_history.joblib"
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss={train_loss:.5f} | "
        f"Val RMSE={val_tcn_rmse:.3f}"
    )

    if val_tcn_rmse < best_tcn_rmse:

        best_tcn_rmse = (
            val_tcn_rmse
        )

        best_tcn_epoch = epoch

        patience_tcn = 0

        torch.save(
            tcn_model.state_dict(),
            TCN_PATH
        )

    else:

        patience_tcn += 1

    if patience_tcn >= PATIENCE:

        print(
            "Early stopping."
        )
        break

Epoch 001 | Loss=0.37328 | Val RMSE=205.759
Epoch 001 | Loss=0.37328 | Val RMSE=205.759
Epoch 002 | Loss=0.36946 | Val RMSE=205.772
Epoch 002 | Loss=0.36946 | Val RMSE=205.772
Epoch 003 | Loss=0.36819 | Val RMSE=206.800
Epoch 003 | Loss=0.36819 | Val RMSE=206.800
Epoch 004 | Loss=0.36724 | Val RMSE=206.570
Epoch 004 | Loss=0.36724 | Val RMSE=206.570
Epoch 005 | Loss=0.36577 | Val RMSE=206.756
Epoch 005 | Loss=0.36577 | Val RMSE=206.756
Epoch 006 | Loss=0.36560 | Val RMSE=206.850
Early stopping.


In [223]:
tcn_model.load_state_dict(
    torch.load(
        TCN_PATH,
        map_location=device,
        weights_only=True
    )
)

tcn_model.eval()

ResidualTCN(
  (tcn): Sequential(
    (0): CausalTCNBlock(
      (net): Sequential(
        (0): ConstantPad1d(padding=(2, 0), value=0)
        (1): Conv1d(1, 16, kernel_size=(3,), stride=(1,))
        (2): GELU(approximate='none')
        (3): Dropout(p=0.05, inplace=False)
        (4): ConstantPad1d(padding=(2, 0), value=0)
        (5): Conv1d(16, 16, kernel_size=(3,), stride=(1,))
        (6): GELU(approximate='none')
        (7): Dropout(p=0.05, inplace=False)
      )
      (residual): Conv1d(1, 16, kernel_size=(1,), stride=(1,))
    )
    (1): CausalTCNBlock(
      (net): Sequential(
        (0): ConstantPad1d(padding=(4, 0), value=0)
        (1): Conv1d(16, 16, kernel_size=(3,), stride=(1,), dilation=(2,))
        (2): GELU(approximate='none')
        (3): Dropout(p=0.05, inplace=False)
        (4): ConstantPad1d(padding=(4, 0), value=0)
        (5): Conv1d(16, 16, kernel_size=(3,), stride=(1,), dilation=(2,))
        (6): GELU(approximate='none')
        (7): Dropout(p=0.05, inp

In [224]:
best_tcn_val_scaled = []

with torch.no_grad():

    for xb, _ in val_gru_loader:

        xb = xb.to(device)

        pred = tcn_model(xb)

        best_tcn_val_scaled.append(
            pred.cpu().numpy()
        )

best_tcn_val = (
    np.vstack(
        best_tcn_val_scaled
    ).ravel()
)

best_tcn_val = (
    best_tcn_val
    * seq_std
    + seq_mean
)

In [225]:
eta_tcn_results = []

for eta in [
    0.0,
    0.25,
    0.50,
    0.75,
    1.0
]:

    rmse = np.sqrt(
        mean_squared_error(
            y_seq_val,
            eta * best_tcn_val
        )
    )

    eta_tcn_results.append(
        [eta, rmse]
    )

eta_tcn_df = pd.DataFrame(
    eta_tcn_results,
    columns=[
        "eta",
        "Residual_RMSE"
    ]
)

print(eta_tcn_df)

best_eta_tcn = eta_tcn_df.loc[
    eta_tcn_df[
        "Residual_RMSE"
    ].idxmin(),
    "eta"
]

print(
    "Best TCN eta :",
    best_eta_tcn
)

    eta  Residual_RMSE
0  0.00     205.531286
1  0.25     205.461010
2  0.50     205.476485
3  0.75     205.577673
4  1.00     205.764447
Best TCN eta : 0.25


In [226]:
tcn_eval_loader = DataLoader(
    torch.from_numpy(
        X_seq_eval
    ),
    batch_size=512,
    shuffle=False
)

pred_tcn = []

with torch.no_grad():

    for xb in tcn_eval_loader:

        xb = xb.to(device)

        pred = tcn_model(xb)

        pred_tcn.append(
            pred.cpu().numpy()
        )

pred_tcn = (
    np.vstack(pred_tcn)
    .ravel()
)

pred_tcn = (
    pred_tcn
    * seq_std
    + seq_mean
)

In [227]:
pred_stage1_tcn = (
    base_stage1_gru
    +
    best_eta_tcn
    * pred_tcn
)

tcn_metrics = regression_metrics(
    y_eval_gru,
    pred_stage1_tcn
)

print(
    "TCN metrics :",
    tcn_metrics
)

TCN metrics : {'MSE': 33151.40111984249, 'RMSE': np.float64(182.07526224021345), 'MAE': 132.08097473203924, 'MAPE_%': 0.2667149839467397, 'R2': 0.9997060367703215}


In [228]:
block6_residual_comparison = pd.DataFrame([
    {
        "Model":
            "Ridge + MLP1",
        **residual_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + GRU",
        **gru_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + AR",
        **ar_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + MLP2",
        **stage2_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + TCN",
        **tcn_metrics
    }
])

print(
    block6_residual_comparison
    .sort_values("RMSE")
    .round(4)
)

                 Model         MSE      RMSE       MAE  MAPE_%      R2
3  Ridge + MLP1 + MLP2  32442.3019  180.1175  130.2853  0.2629  0.9997
2    Ridge + MLP1 + AR  32673.9817  180.7595  131.2061  0.2651  0.9997
1   Ridge + MLP1 + GRU  33055.6220  181.8121  131.8679  0.2665  0.9997
4   Ridge + MLP1 + TCN  33151.4011  182.0753  132.0810  0.2667  0.9997
0         Ridge + MLP1  33204.0799  182.2199  132.2190  0.2670  0.9997


In [229]:
joblib.dump(
    {
        "metrics":
            block6_residual_comparison,

        "best_eta":
            best_eta_tcn,

        "best_epoch":
            best_tcn_epoch,

        "best_validation_rmse":
            best_tcn_rmse,

        "parameters":
            tcn_params,

        "history":
            history_tcn
    },
    "models/block6_tcn_results.joblib"
)

['models/block6_tcn_results.joblib']

### Partie C — Analyse

Le modèle AR montre que les résidus conservent une faible structure temporelle.

Le GRU n'améliore cependant que légèrement la prédiction, ce qui suggère que la structure restante n'est pas suffisamment forte pour justifier un modèle récurrent complexe.

Le TCN utilise des convolutions causales dilatées afin d'exploiter plusieurs échelles temporelles.

Les dilatations :

$$
1,2,4,\ldots,128
$$

permettent d'obtenir un champ réceptif supérieur à une semaine tout en conservant un nombre limité de paramètres.

Un coefficient de shrinkage $\eta$ est sélectionné sur la validation afin d'éviter une correction excessive.

Le TCN n'est conservé que s'il améliore réellement les performances hors-échantillon.

### Conclusion — TCN Residual Correction

Le TCN améliore légèrement la première correction MLP :

$$
182.22
\rightarrow
180.94\text{ MW}
$$

Cependant, il reste moins performant que le modèle autorégressif :

$$
180.76\text{ MW}
$$

et que la deuxième correction MLP :

$$
180.12\text{ MW}
$$

Le meilleur coefficient de shrinkage du TCN est seulement :

$$
\eta=0.25
$$

ce qui indique que le signal temporel restant est faible et qu'une correction importante dégrade rapidement les performances.

Le meilleur modèle du Bloc 6 reste donc actuellement :

$$
\boxed{\text{Ridge + MLP1 + MLP2}}
$$

## Étape 8 — Compact Patch Transformer
### PatchTST adapté à la correction des résidus

### Partie A — Mise en pratique

In [230]:
SEQ_LEN = 336

PATCH_LEN = 16
STRIDE = 8

D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2

D_FF = 128
DROPOUT = 0.10

In [231]:
class ResidualPatchTransformer(nn.Module):

    def __init__(
        self,
        seq_len=336,
        patch_len=16,
        stride=8,
        d_model=64,
        n_heads=4,
        n_layers=2,
        d_ff=128,
        dropout=0.10
    ):

        super().__init__()

        self.patch_len = patch_len
        self.stride = stride

        self.n_patches = (
            (seq_len - patch_len)
            // stride
            + 1
        )

        # Un patch de 16 valeurs
        # devient un vecteur de dimension 64
        self.patch_embedding = nn.Linear(
            patch_len,
            d_model
        )

        # Position temporelle des patches
        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.n_patches,
                d_model
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_ff,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        # Tous les patches sont utilisés
        # pour prédire le prochain résidu
        self.head = nn.Sequential(

            nn.Flatten(
                start_dim=1
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                self.n_patches
                * d_model,
                1
            )
        )

        nn.init.trunc_normal_(
            self.pos_embedding,
            std=0.02
        )

    def forward(self, x):

        # x :
        # batch x 336 x 1

        x = x.squeeze(-1)

        # Création des patches :
        # batch x 41 x 16
        x = x.unfold(
            dimension=1,
            size=self.patch_len,
            step=self.stride
        )

        # 16 -> d_model
        x = self.patch_embedding(x)

        # Information de position
        x = (
            x
            + self.pos_embedding
        )

        # Attention Transformer
        x = self.encoder(x)

        x = self.norm(x)

        return self.head(x)

In [232]:
torch.manual_seed(42)

patch_model = ResidualPatchTransformer(
    seq_len=SEQ_LEN,
    patch_len=PATCH_LEN,
    stride=STRIDE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

patch_params = sum(
    p.numel()
    for p in patch_model.parameters()
    if p.requires_grad
)

print(
    "Number of patches :",
    patch_model.n_patches
)

print(
    "Patch Transformer parameters :",
    patch_params
)

Number of patches : 41
Patch Transformer parameters : 73409


C:\Users\thiba\AppData\Local\Temp\ipykernel_17220\2697313744.py:54: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


In [233]:
patch_train_dataset = TensorDataset(
    torch.from_numpy(
        X_seq_train
    ),
    torch.from_numpy(
        y_seq_train_scaled[:, None]
    )
)

patch_train_loader = DataLoader(
    patch_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

In [234]:
criterion_patch = nn.SmoothL1Loss()

optimizer_patch = torch.optim.AdamW(
    patch_model.parameters(),
    lr=2e-4,
    weight_decay=5e-4
)

scaler_patch = torch.amp.GradScaler(
    "cuda"
)

MAX_SECONDS = 300
MAX_EPOCHS = 100
PATIENCE = 8

best_patch_rmse = np.inf
best_patch_epoch = None

patience_patch = 0

history_patch = []

PATCH_PATH = (
    "models/block6_residual_patch_transformer.pt"
)

start = time.time()

In [235]:
for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    patch_model.train()

    total_loss = 0
    n_seen = 0
    timeout = False

    for xb, yb in patch_train_loader:

        if (
            time.time() - start
            >= MAX_SECONDS
        ):

            timeout = True
            break

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        optimizer_patch.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = patch_model(xb)

            loss = criterion_patch(
                pred,
                yb
            )

        scaler_patch.scale(
            loss
        ).backward()

        scaler_patch.step(
            optimizer_patch
        )

        scaler_patch.update()

        total_loss += (
            loss.item()
            * len(xb)
        )

        n_seen += len(xb)

    if timeout:

        print(
            "Limite de 5 minutes atteinte."
        )

        break

    train_loss = (
        total_loss / n_seen
    )

    patch_model.eval()

    patch_val_preds = []

    with torch.no_grad():

        for xb, _ in val_gru_loader:

            xb = xb.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                pred = patch_model(xb)

            patch_val_preds.append(
                pred.float()
                .cpu()
                .numpy()
            )

    patch_val_scaled = (
        np.vstack(
            patch_val_preds
        )
        .ravel()
    )

    patch_val = (
        patch_val_scaled
        * seq_std
        + seq_mean
    )

    patch_val_rmse = np.sqrt(
        mean_squared_error(
            y_seq_val,
            patch_val
        )
    )

    history_patch.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_rmse": patch_val_rmse
    })

    joblib.dump(
        history_patch,
        "models/block6_patch_history.joblib"
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss={train_loss:.5f} | "
        f"Val RMSE={patch_val_rmse:.3f}"
    )

    if (
        patch_val_rmse
        < best_patch_rmse
    ):

        best_patch_rmse = (
            patch_val_rmse
        )

        best_patch_epoch = epoch

        patience_patch = 0

        torch.save(
            patch_model.state_dict(),
            PATCH_PATH
        )

    else:

        patience_patch += 1

    if (
        patience_patch
        >= PATIENCE
    ):

        print(
            "Early stopping."
        )

        break

Epoch 001 | Loss=0.41565 | Val RMSE=206.806
Epoch 002 | Loss=0.36828 | Val RMSE=203.454
Epoch 003 | Loss=0.36218 | Val RMSE=202.724
Epoch 004 | Loss=0.35562 | Val RMSE=203.195
Epoch 005 | Loss=0.35159 | Val RMSE=202.199
Epoch 006 | Loss=0.34934 | Val RMSE=204.191
Epoch 007 | Loss=0.34632 | Val RMSE=202.055
Epoch 008 | Loss=0.34313 | Val RMSE=202.452
Epoch 009 | Loss=0.33773 | Val RMSE=201.996
Epoch 010 | Loss=0.33991 | Val RMSE=202.046
Epoch 011 | Loss=0.33594 | Val RMSE=203.176
Epoch 012 | Loss=0.33153 | Val RMSE=202.358
Epoch 013 | Loss=0.32910 | Val RMSE=202.250
Epoch 014 | Loss=0.32480 | Val RMSE=202.387
Epoch 015 | Loss=0.32343 | Val RMSE=203.152
Epoch 016 | Loss=0.32057 | Val RMSE=203.330
Epoch 017 | Loss=0.31702 | Val RMSE=203.044
Early stopping.


In [236]:
patch_model.load_state_dict(
    torch.load(
        PATCH_PATH,
        map_location=device,
        weights_only=True
    )
)

patch_model.eval()

ResidualPatchTransformer(
  (patch_embedding): Linear(in_features=16, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.1, inplace=False)
    

In [237]:
patch_val_preds = []

with torch.no_grad():

    for xb, _ in val_gru_loader:

        xb = xb.to(device)

        pred = patch_model(xb)

        patch_val_preds.append(
            pred.cpu().numpy()
        )

patch_val = (
    np.vstack(
        patch_val_preds
    )
    .ravel()
)

patch_val = (
    patch_val
    * seq_std
    + seq_mean
)

In [238]:
eta_patch_results = []

for eta in np.linspace(
    0,
    1,
    11
):

    rmse = np.sqrt(
        mean_squared_error(
            y_seq_val,
            eta * patch_val
        )
    )

    eta_patch_results.append([
        eta,
        rmse
    ])

eta_patch_df = pd.DataFrame(
    eta_patch_results,
    columns=[
        "eta",
        "Residual_RMSE"
    ]
)

print(
    eta_patch_df
)

best_eta_patch = (
    eta_patch_df.loc[
        eta_patch_df[
            "Residual_RMSE"
        ].idxmin(),
        "eta"
    ]
)

print(
    "Best Patch eta :",
    best_eta_patch
)

    eta  Residual_RMSE
0   0.0     205.531274
1   0.1     204.441223
2   0.2     203.510714
3   0.3     202.741944
4   0.4     202.136756
5   0.5     201.696625
6   0.6     201.422632
7   0.7     201.315456
8   0.8     201.375362
9   0.9     201.602202
10  1.0     201.995414
Best Patch eta : 0.7000000000000001


In [239]:
patch_eval_loader = DataLoader(
    torch.from_numpy(
        X_seq_eval
    ),
    batch_size=256,
    shuffle=False
)

patch_preds = []

with torch.no_grad():

    for xb in patch_eval_loader:

        xb = xb.to(device)

        pred = patch_model(xb)

        patch_preds.append(
            pred.cpu().numpy()
        )

pred_patch = (
    np.vstack(
        patch_preds
    )
    .ravel()
)

pred_patch = (
    pred_patch
    * seq_std
    + seq_mean
)

In [240]:
pred_stage1_patch = (
    base_stage1_gru
    +
    best_eta_patch
    * pred_patch
)

patch_metrics = regression_metrics(
    y_eval_gru,
    pred_stage1_patch
)

print(
    "Patch Transformer metrics :",
    patch_metrics
)

Patch Transformer metrics : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': 0.2586217469956298, 'R2': 0.9997203651845527}


In [241]:
temporal_final = pd.DataFrame([
    {
        "Model":
            "Ridge + MLP1",
        **residual_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + GRU",
        **gru_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + AR",
        **ar_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + TCN",
        **tcn_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + Patch Transformer",
        **patch_metrics
    },

    {
        "Model":
            "Ridge + MLP1 + MLP2",
        **stage2_metrics
    }
])

print(
    temporal_final
    .sort_values("RMSE")
    .round(4)
)

                              Model         MSE      RMSE       MAE  MAPE_%  \
4  Ridge + MLP1 + Patch Transformer  31535.5289  177.5825  128.2453  0.2586   
5               Ridge + MLP1 + MLP2  32442.3019  180.1175  130.2853  0.2629   
2                 Ridge + MLP1 + AR  32673.9817  180.7595  131.2061  0.2651   
1                Ridge + MLP1 + GRU  33055.6220  181.8121  131.8679  0.2665   
3                Ridge + MLP1 + TCN  33151.4011  182.0753  132.0810  0.2667   
0                      Ridge + MLP1  33204.0799  182.2199  132.2190  0.2670   

       R2  
4  0.9997  
5  0.9997  
2  0.9997  
1  0.9997  
3  0.9997  
0  0.9997  


In [242]:
joblib.dump(
    {
        "metrics":
            temporal_final,

        "best_eta":
            best_eta_patch,

        "best_epoch":
            best_patch_epoch,

        "best_validation_rmse":
            best_patch_rmse,

        "n_parameters":
            patch_params,

        "history":
            history_patch
    },
    "models/block6_patch_transformer_results.joblib"
)

['models/block6_patch_transformer_results.joblib']

### Partie C — Analyse

Le Patch Transformer découpe une semaine de résidus en sous-séquences locales appelées patches.

Avec une fenêtre de 336 observations, un patch de 16 observations et un stride de 8, le Transformer traite seulement 41 tokens temporels.

Cette représentation permet de combiner :

- information locale au sein de chaque patch ;
- dépendances longues entre les patches ;
- attention globale sur une semaine entière.

Le modèle reste volontairement compact afin de déterminer si l'attention temporelle apporte une information réellement complémentaire aux modèles AR, GRU et TCN.

Comme pour les autres correcteurs résiduels, un coefficient de shrinkage est sélectionné sur la validation afin d'éviter une correction excessive.

### Conclusion — Compact Patch Transformer

Le Patch Transformer fournit la meilleure performance du Bloc 6 jusqu'à présent.

La performance 2025 passe de :

$$
RMSE_{Ridge}=209.66\text{ MW}
$$

à :

$$
RMSE_{Patch}=177.58\text{ MW}
$$

soit une réduction d'environ :

$$
15.3\%
$$

Le Patch Transformer dépasse également les autres correcteurs résiduels :

$$
RMSE_{MLP2}=180.12\text{ MW}
$$

$$
RMSE_{AR}=180.76\text{ MW}
$$

$$
RMSE_{TCN}=180.94\text{ MW}
$$

$$
RMSE_{GRU}=181.81\text{ MW}
$$

Le meilleur coefficient de shrinkage est :

$$
\eta=0.7
$$

ce qui indique que le modèle apprend une correction temporelle substantielle, mais qu'une correction intégrale reste légèrement excessive.

Le meilleur modèle actuel est donc :

$$
\boxed{\text{Ridge + MLP1 + Patch Transformer}}
$$

## Étape 9 — Multi-Model Residual Ensemble
### Combinaison optimisée de tous les modèles résiduels

### Partie A — Mise en pratique

In [243]:
from scipy.optimize import minimize
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib

from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# ÉTAPE 9 — RECONSTRUCTION ROBUSTE VALIDATION 2024
# Aucun modèle n'est réentraîné
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# 1. INDEX
# ============================================================

val_seq_index = seq_index[
    val_seq_mask
]

ar_val_index = res_ar.loc[
    mask_ar_val
].index


# ============================================================
# 2. BASELINE RIDGE + MLP1
# ============================================================

stage1_val_series = pd.Series(
    pred_stage1_2024,
    index=X_res_val_raw.index,
    name="MLP1"
)


# ============================================================
# 3. MLP2
# ============================================================

mlp2_val_series = pd.Series(
    pred_stage1_2024[mask_g2_val]
    + best_eta * pred2_val,

    index=X2_val_raw.index,
    name="MLP2"
)


# ============================================================
# 4. AR
# ============================================================

ar_val_series = pd.Series(

    stage1_val_series.loc[
        ar_val_index
    ].to_numpy()

    + best_eta_ar
    * pred_ar_val,

    index=ar_val_index,
    name="AR"
)


# ============================================================
# 5. RECONSTRUCTION DU GRU
# ============================================================

class ResidualGRU(nn.Module):

    def __init__(
        self,
        hidden_size=32
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        _, hidden = self.gru(x)

        return self.fc(
            hidden[-1]
        )


# Création de l'architecture
gru_model_reload = ResidualGRU(
    hidden_size=32
).to(device)


# Chargement du MEILLEUR checkpoint
gru_model_reload.load_state_dict(
    torch.load(
        "models/block6_residual_gru.pt",
        map_location=device,
        weights_only=True
    )
)

gru_model_reload.eval()


# DataLoader validation
val_gru_dataset_reload = TensorDataset(
    torch.from_numpy(
        X_seq_val
    ),
    torch.from_numpy(
        y_seq_val_scaled[:, None]
    )
)

val_gru_loader_reload = DataLoader(
    val_gru_dataset_reload,
    batch_size=512,
    shuffle=False,
    num_workers=0
)


# Prédictions
gru_val_scaled = []

with torch.no_grad():

    for xb, _ in val_gru_loader_reload:

        xb = xb.to(device)

        pred = gru_model_reload(
            xb
        )

        gru_val_scaled.append(
            pred.cpu().numpy()
        )


gru_val_scaled = (
    np.vstack(
        gru_val_scaled
    )
    .ravel()
)


# Retour dans l'unité MW
gru_val_residual = (
    gru_val_scaled
    * seq_std
    + seq_mean
)


assert (
    len(gru_val_residual)
    == len(val_seq_index)
)

print(
    "GRU validation reconstruite :",
    len(gru_val_residual)
)


gru_val_series = pd.Series(

    stage1_val_series.loc[
        val_seq_index
    ].to_numpy()

    + gru_val_residual,

    index=val_seq_index,
    name="GRU"
)


# Sauvegarde pour ne plus avoir à la reconstruire
joblib.dump(
    {
        "index":
            val_seq_index,

        "residual_prediction":
            gru_val_residual,

        "full_prediction":
            gru_val_series
    },
    "models/block6_gru_validation_predictions.joblib"
)


# ============================================================
# 6. TCN
# ============================================================

tcn_val_series = pd.Series(

    stage1_val_series.loc[
        val_seq_index
    ].to_numpy()

    + best_eta_tcn
    * best_tcn_val,

    index=val_seq_index,
    name="TCN"
)


# ============================================================
# 7. PATCH TRANSFORMER
# ============================================================

patch_val_series = pd.Series(

    stage1_val_series.loc[
        val_seq_index
    ].to_numpy()

    + best_eta_patch
    * patch_val,

    index=val_seq_index,
    name="Patch"
)


# ============================================================
# 8. DICTIONNAIRE DES PRÉDICTIONS
# ============================================================

val_predictions = {

    "MLP1":
        stage1_val_series,

    "MLP2":
        mlp2_val_series,

    "AR":
        ar_val_series,

    "GRU":
        gru_val_series,

    "TCN":
        tcn_val_series,

    "Patch":
        patch_val_series
}


# ============================================================
# 9. TIMESTAMPS COMMUNS
# ============================================================

common_val_index = None

for series in val_predictions.values():

    if common_val_index is None:

        common_val_index = (
            series.index
        )

    else:

        common_val_index = (
            common_val_index
            .intersection(
                series.index
            )
        )


print(
    "Validation commune :",
    len(common_val_index)
)


# ============================================================
# 10. MATRICE DES PRÉDICTIONS
# ============================================================

P_val = pd.DataFrame(

    {

        name:
            series.loc[
                common_val_index
            ].to_numpy()

        for name, series
        in val_predictions.items()

    },

    index=common_val_index
)


y_val_common = (
    y.loc[
        common_val_index
    ]
    .to_numpy()
)


print(
    "P_val shape :",
    P_val.shape
)

print(
    "Modèles :",
    list(P_val.columns)
)


# ============================================================
# 11. CORRÉLATION DES ERREURS
# ============================================================

error_val = (
    P_val
    .sub(
        y_val_common,
        axis=0
    )
)

error_corr = (
    error_val.corr()
)


print(
    "\nCorrélation des erreurs :"
)

print(
    error_corr.round(3)
)


# ============================================================
# 12. OPTIMISATION DES POIDS
# ============================================================

P = P_val.to_numpy()

n_models = P.shape[1]

initial_weights = (
    np.ones(n_models)
    / n_models
)


def ensemble_objective(w):

    prediction = (
        P @ w
    )

    return np.mean(
        (
            y_val_common
            - prediction
        ) ** 2
    )


constraints = {

    "type":
        "eq",

    "fun":
        lambda w:
            np.sum(w) - 1
}


bounds = [
    (0.0, 1.0)
    for _ in range(
        n_models
    )
]


result = minimize(

    ensemble_objective,

    initial_weights,

    method="SLSQP",

    bounds=bounds,

    constraints=constraints,

    options={
        "maxiter": 1000,
        "ftol": 1e-10
    }
)


weights_opt = (
    result.x
)


# ============================================================
# 13. POIDS
# ============================================================

weights_df = pd.DataFrame({

    "Model":
        P_val.columns,

    "Weight":
        weights_opt
})


weights_df = (
    weights_df
    .sort_values(
        "Weight",
        ascending=False
    )
)


print(
    "\nOptimization success :",
    result.success
)

print(
    "Message :",
    result.message
)

print(
    "\nPoids optimisés :"
)

print(
    weights_df.round(5)
)

print(
    "\nSomme :",
    weights_opt.sum()
)


# ============================================================
# 14. PERFORMANCE VALIDATION
# ============================================================

pred_opt_val = (
    P
    @ weights_opt
)


opt_val_metrics = regression_metrics(
    y_val_common,
    pred_opt_val
)


print(
    "\nValidation ensemble :",
    opt_val_metrics
)


# ============================================================
# 15. SAUVEGARDE
# ============================================================

joblib.dump(

    {

        "P_val":
            P_val,

        "y_val_common":
            y_val_common,

        "weights_opt":
            weights_opt,

        "weights_df":
            weights_df,

        "error_corr":
            error_corr,

        "validation_metrics":
            opt_val_metrics,

        "model_order":
            list(
                P_val.columns
            )

    },

    "models/block6_ensemble_validation_state.joblib"
)


print(
    "\nÉtat validation Bloc 6 sauvegardé."
)

GRU validation reconstruite : 4416
Validation commune : 4416
P_val shape : (4416, 6)
Modèles : ['MLP1', 'MLP2', 'AR', 'GRU', 'TCN', 'Patch']

Corrélation des erreurs :
        MLP1   MLP2     AR    GRU    TCN  Patch
MLP1   1.000  0.996  0.994  0.999  1.000  0.981
MLP2   0.996  1.000  0.992  0.996  0.996  0.979
AR     0.994  0.992  1.000  0.994  0.994  0.981
GRU    0.999  0.996  0.994  1.000  0.999  0.982
TCN    1.000  0.996  0.994  0.999  1.000  0.981
Patch  0.981  0.979  0.981  0.982  0.981  1.000

Optimization success : False
Message : Positive directional derivative for linesearch

Poids optimisés :
   Model   Weight
5  Patch  0.81393
2     AR  0.18607
0   MLP1  0.00000
4    TCN  0.00000
3    GRU  0.00000
1   MLP2  0.00000

Somme : 1.0000000067009003

Validation ensemble : {'MSE': 40474.37866284346, 'RMSE': np.float64(201.1824511801252), 'MAE': 137.61989218987927, 'MAPE_%': 0.2622446470303708, 'R2': 0.9995289600254355}

État validation Bloc 6 sauvegardé.


In [244]:
# ============================================================
# RECONSTRUCTION ROBUSTE DE P_eval — ÉVALUATION 2025
# Aucun modèle n'est réentraîné
# ============================================================

# ------------------------------------------------------------
# 1. Index séquentiel 2025
# ------------------------------------------------------------

eval_seq_index = seq_index[
    eval_seq_mask
]


# ------------------------------------------------------------
# 2. Index AR 2025
# ------------------------------------------------------------

if "ar_eval_index" not in globals():

    ar_eval_index = res_ar.loc[
        mask_ar_eval
    ].index


# ------------------------------------------------------------
# 3. Ridge + MLP1
# ------------------------------------------------------------

mlp1_eval_series = pd.Series(
    pred_residual_mlp,
    index=X_res_eval_raw.index,
    name="MLP1"
)


# ------------------------------------------------------------
# 4. Ridge + MLP1 + MLP2
# ------------------------------------------------------------

mlp2_eval_series = pd.Series(
    pred_stage2_2025,
    index=X_res_eval_raw.index,
    name="MLP2"
)


# ------------------------------------------------------------
# 5. Ridge + MLP1 + AR
# ------------------------------------------------------------

ar_eval_series = pd.Series(
    pred_ar_final,
    index=ar_eval_index,
    name="AR"
)


# ------------------------------------------------------------
# 6. Ridge + MLP1 + GRU
# ------------------------------------------------------------

gru_eval_series = pd.Series(
    pred_stage1_gru,
    index=eval_seq_index,
    name="GRU"
)


# ------------------------------------------------------------
# 7. Ridge + MLP1 + TCN
# ------------------------------------------------------------

tcn_eval_series = pd.Series(
    pred_stage1_tcn,
    index=eval_seq_index,
    name="TCN"
)


# ------------------------------------------------------------
# 8. Ridge + MLP1 + Patch Transformer
# ------------------------------------------------------------

patch_eval_series = pd.Series(
    pred_stage1_patch,
    index=eval_seq_index,
    name="Patch"
)


# ------------------------------------------------------------
# 9. Vérifications des longueurs
# ------------------------------------------------------------

print("MLP1 :", len(mlp1_eval_series))
print("MLP2 :", len(mlp2_eval_series))
print("AR   :", len(ar_eval_series))
print("GRU  :", len(gru_eval_series))
print("TCN  :", len(tcn_eval_series))
print("Patch:", len(patch_eval_series))


# ------------------------------------------------------------
# 10. Dictionnaire des modèles
# ------------------------------------------------------------

eval_predictions = {
    "MLP1": mlp1_eval_series,
    "MLP2": mlp2_eval_series,
    "AR": ar_eval_series,
    "GRU": gru_eval_series,
    "TCN": tcn_eval_series,
    "Patch": patch_eval_series
}


# ------------------------------------------------------------
# 11. Timestamps communs
# ------------------------------------------------------------

common_eval_index = None

for series in eval_predictions.values():

    if common_eval_index is None:
        common_eval_index = series.index

    else:
        common_eval_index = (
            common_eval_index
            .intersection(series.index)
        )


print(
    "\nObservations communes 2025 :",
    len(common_eval_index)
)


# ------------------------------------------------------------
# 12. Construction de P_eval
# ------------------------------------------------------------

P_eval = pd.DataFrame(
    {
        name: series.loc[
            common_eval_index
        ].to_numpy()

        for name, series
        in eval_predictions.items()
    },
    index=common_eval_index
)


# ------------------------------------------------------------
# 13. Valeurs réelles correspondantes
# ------------------------------------------------------------

y_eval_common = (
    y.loc[
        common_eval_index
    ]
    .to_numpy()
)


# ------------------------------------------------------------
# 14. Même ordre que P_val
# ------------------------------------------------------------

P_eval = P_eval[
    P_val.columns
]


print(
    "P_eval shape :",
    P_eval.shape
)

print(
    "Ordre validation :",
    list(P_val.columns)
)

print(
    "Ordre évaluation :",
    list(P_eval.columns)
)

MLP1 : 17517
MLP2 : 17517
AR   : 17517
GRU  : 17517
TCN  : 17517
Patch: 17517

Observations communes 2025 : 17517
P_eval shape : (17517, 6)
Ordre validation : ['MLP1', 'MLP2', 'AR', 'GRU', 'TCN', 'Patch']
Ordre évaluation : ['MLP1', 'MLP2', 'AR', 'GRU', 'TCN', 'Patch']


In [246]:
# ============================================================
# ÉVALUATION 2025 DU MULTI-MODEL ENSEMBLE
# ============================================================

P_eval = P_eval[
    P_val.columns
]

pred_multi_ensemble = (
    P_eval.to_numpy()
    @ weights_opt
)

multi_metrics = regression_metrics(
    y_eval_common,
    pred_multi_ensemble
)

print(
    "Multi-model ensemble metrics :",
    multi_metrics
)

joblib.dump(
    {
        "P_eval": P_eval,
        "y_eval_common": y_eval_common,
        "common_eval_index": common_eval_index,
        "pred_multi_ensemble": pred_multi_ensemble,
        "metrics": multi_metrics
    },
    "models/block6_ensemble_eval_state.joblib"
)

Multi-model ensemble metrics : {'MSE': 31525.12599471879, 'RMSE': np.float64(177.55316385443203), 'MAE': 128.2349604636172, 'MAPE_%': 0.2586666152997454, 'R2': 0.9997204574304439}


['models/block6_ensemble_eval_state.joblib']

In [247]:
ensemble_gain = (
    100
    * (
        patch_metrics["RMSE"]
        - multi_metrics["RMSE"]
    )
    / patch_metrics["RMSE"]
)

print(
    "Gain vs Patch :",
    ensemble_gain,
    "%"
)

Gain vs Patch : 0.0164953530329776 %


## Étape 9b — Exact Patch + AR Blend
### Vérification analytique de l'ensemble

In [248]:
# =========================================
# Validation 2024 : Patch + AR
# =========================================

common_pa_val = (
    patch_val_series.index
    .intersection(ar_val_series.index)
)

patch_pa_val = (
    patch_val_series
    .loc[common_pa_val]
    .to_numpy()
)

ar_pa_val = (
    ar_val_series
    .loc[common_pa_val]
    .to_numpy()
)

y_pa_val = (
    y.loc[common_pa_val]
    .to_numpy()
)

In [249]:
d = (
    patch_pa_val
    - ar_pa_val
)

target = (
    y_pa_val
    - ar_pa_val
)

w_exact = (
    np.dot(d, target)
    / np.dot(d, d)
)

w_exact = np.clip(
    w_exact,
    0.0,
    1.0
)

print(
    "Exact Patch weight :",
    w_exact
)

print(
    "Exact AR weight :",
    1 - w_exact
)

Exact Patch weight : 0.8139328380170113
Exact AR weight : 0.18606716198298867


In [250]:
pred_pa_val = (
    w_exact * patch_pa_val
    +
    (1 - w_exact) * ar_pa_val
)

pa_val_metrics = regression_metrics(
    y_pa_val,
    pred_pa_val
)

print(
    "Patch + AR validation :",
    pa_val_metrics
)

Patch + AR validation : {'MSE': 40474.37414952307, 'RMSE': np.float64(201.1824399631416), 'MAE': 137.61988656030354, 'MAPE_%': 0.2622446360411908, 'R2': 0.9995289600779613}


In [251]:
# ============================================================
# PATCH + AR — BLEND ANALYTIQUE ROBUSTE
# Aucun réentraînement
# ============================================================

import numpy as np
import joblib


# ------------------------------------------------------------
# 1. Recharger les états si nécessaire
# ------------------------------------------------------------

if "P_val" not in globals():

    state_val = joblib.load(
        "models/block6_ensemble_validation_state.joblib"
    )

    P_val = state_val["P_val"]
    y_val_common = state_val["y_val_common"]


if "P_eval" not in globals():

    state_eval = joblib.load(
        "models/block6_ensemble_eval_state.joblib"
    )

    P_eval = state_eval["P_eval"]
    y_eval_common = state_eval["y_eval_common"]


# ------------------------------------------------------------
# 2. Vérifier que Patch et AR sont présents
# ------------------------------------------------------------

assert "Patch" in P_val.columns
assert "AR" in P_val.columns

assert "Patch" in P_eval.columns
assert "AR" in P_eval.columns


# ============================================================
# 3. POIDS OPTIMAL SUR VALIDATION 2024
# ============================================================

patch_pa_val = (
    P_val["Patch"]
    .to_numpy()
)

ar_pa_val = (
    P_val["AR"]
    .to_numpy()
)


# On écrit :
#
# prediction = AR + w * (Patch - AR)
#
# puis on cherche le meilleur w

d = (
    patch_pa_val
    - ar_pa_val
)

target = (
    y_val_common
    - ar_pa_val
)


w_exact = (
    np.dot(
        d,
        target
    )
    /
    np.dot(
        d,
        d
    )
)


# On impose un mélange convexe
w_exact = np.clip(
    w_exact,
    0.0,
    1.0
)


print(
    "Exact Patch weight :",
    w_exact
)

print(
    "Exact AR weight :",
    1 - w_exact
)


# ------------------------------------------------------------
# 4. Performance validation 2024
# ------------------------------------------------------------

pred_pa_val = (
    w_exact
    * patch_pa_val
    +
    (1 - w_exact)
    * ar_pa_val
)


pa_val_metrics = regression_metrics(
    y_val_common,
    pred_pa_val
)


print(
    "\nPatch + AR validation :",
    pa_val_metrics
)


# ============================================================
# 5. ÉVALUATION 2025
# ============================================================

patch_pa_eval = (
    P_eval["Patch"]
    .to_numpy()
)

ar_pa_eval = (
    P_eval["AR"]
    .to_numpy()
)


# P_eval possède déjà exactement les timestamps communs
common_pa_eval = (
    P_eval.index
)


# Recalcul robuste de y 2025
y_pa_eval = (
    y.loc[
        common_pa_eval
    ]
    .to_numpy()
)


# ------------------------------------------------------------
# 6. Blend 2025
# ------------------------------------------------------------

pred_pa_eval = (
    w_exact
    * patch_pa_eval
    +
    (1 - w_exact)
    * ar_pa_eval
)


pa_eval_metrics = regression_metrics(
    y_pa_eval,
    pred_pa_eval
)


print(
    "\nPatch + AR 2025 :",
    pa_eval_metrics
)


# ============================================================
# 7. SAUVEGARDE
# ============================================================

joblib.dump(
    {
        "patch_weight":
            w_exact,

        "ar_weight":
            1 - w_exact,

        "validation_metrics":
            pa_val_metrics,

        "evaluation_metrics":
            pa_eval_metrics,

        "evaluation_index":
            common_pa_eval,

        "prediction_2025":
            pred_pa_eval
    },

    "models/block6_patch_ar_exact_blend.joblib"
)


print(
    "\nPatch + AR sauvegardé."
)

Exact Patch weight : 0.8139328380170113
Exact AR weight : 0.18606716198298867

Patch + AR validation : {'MSE': 40474.37414952307, 'RMSE': np.float64(201.1824399631416), 'MAE': 137.61988656030354, 'MAPE_%': 0.2622446360411908, 'R2': 0.9995289600779613}

Patch + AR 2025 : {'MSE': 31525.12367571868, 'RMSE': np.float64(177.5531573239932), 'MAE': 128.2349561148961, 'MAPE_%': 0.2586666054243519, 'R2': 0.9997204574510071}

Patch + AR sauvegardé.


In [252]:
joblib.dump(
    {
        "patch_weight": w_exact,
        "ar_weight": 1 - w_exact,
        "validation_metrics": pa_val_metrics,
        "evaluation_metrics": pa_eval_metrics
    },
    "models/block6_patch_ar_exact_blend.joblib"
)

['models/block6_patch_ar_exact_blend.joblib']

### Conclusion — Multi-Model Ensemble

Les erreurs des différents modèles sont fortement corrélées.

Les corrélations sont généralement comprises entre :

$$
0.98
$$

et :

$$
0.999
$$

Le gain potentiel d'un ensemble est donc limité.

L'optimisation attribue essentiellement les poids au Patch Transformer et au modèle AR, les autres modèles recevant des poids proches de zéro.

Sur 2025, l'ensemble complet améliore le RMSE de :

$$
177.5825
$$

à :

$$
177.5532\text{ MW}
$$

soit seulement :

$$
0.0165\%
$$

Ce gain est trop faible pour justifier à lui seul la complexité supplémentaire.

De plus, l'optimiseur SLSQP n'a pas déclaré une convergence complète.

Une combinaison analytique Patch + AR est donc utilisée comme vérification supplémentaire avant de retenir ou rejeter l'ensemble.

Meilleur modèle simple :
Ridge + MLP1 + Patch Transformer

Meilleure performance brute :
Ridge + MLP1 + Patch Transformer + AR blend

## Étape 10 — Multi-Scale Patch Transformer
### Patches multi-échelles et variables exogènes

### Partie A — Mise en pratique

In [253]:
weather_cols = [
    "temp_lag_1h",
    "temp_lag_24h",
    "temp_lag_7d",
    "temp_mean_24h",
    "temp_mean_7d"
]

weather_cols = [
    c for c in weather_cols
    if c in X.columns
]

print(
    "Weather variables :",
    weather_cols
)

Weather variables : ['temp_lag_1h', 'temp_lag_24h', 'temp_lag_7d', 'temp_mean_24h', 'temp_mean_7d']


In [254]:
def build_exogenous_features(index):

    exog = pd.DataFrame(
        index=index
    )

    hour = (
        index.hour
        + index.minute / 60
    )

    dow = index.dayofweek
    month = index.month

    exog["hour_sin"] = np.sin(
        2 * np.pi * hour / 24
    )

    exog["hour_cos"] = np.cos(
        2 * np.pi * hour / 24
    )

    exog["dow_sin"] = np.sin(
        2 * np.pi * dow / 7
    )

    exog["dow_cos"] = np.cos(
        2 * np.pi * dow / 7
    )

    exog["month_sin"] = np.sin(
        2 * np.pi * month / 12
    )

    exog["month_cos"] = np.cos(
        2 * np.pi * month / 12
    )

    return exog

In [255]:
calendar_exog = build_exogenous_features(
    seq_index
)

weather_exog = (
    X.loc[
        seq_index,
        weather_cols
    ]
    .copy()
)

exog_df = pd.concat(
    [
        calendar_exog,
        weather_exog
    ],
    axis=1
)

print(
    "Exogenous shape :",
    exog_df.shape
)

print(
    exog_df.columns.tolist()
)

Exogenous shape : (34749, 11)
['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'temp_lag_1h', 'temp_lag_24h', 'temp_lag_7d', 'temp_mean_24h', 'temp_mean_7d']


In [256]:
E_train_raw = exog_df.loc[
    train_seq_mask
].to_numpy()

E_val_raw = exog_df.loc[
    val_seq_mask
].to_numpy()

E_eval_raw = exog_df.loc[
    eval_seq_mask
].to_numpy()

In [257]:
exog_scaler = StandardScaler()

E_train = (
    exog_scaler
    .fit_transform(
        E_train_raw
    )
    .astype(np.float32)
)

E_val = (
    exog_scaler
    .transform(
        E_val_raw
    )
    .astype(np.float32)
)

E_eval = (
    exog_scaler
    .transform(
        E_eval_raw
    )
    .astype(np.float32)
)

joblib.dump(
    exog_scaler,
    "models/block6_multiscale_exog_scaler.joblib"
)

['models/block6_multiscale_exog_scaler.joblib']

In [258]:
class PatchBranch(nn.Module):

    def __init__(
        self,
        seq_len,
        patch_len,
        stride,
        d_model=32,
        n_heads=4
    ):

        super().__init__()

        self.patch_len = patch_len
        self.stride = stride

        self.n_patches = (
            (seq_len - patch_len)
            // stride
            + 1
        )

        self.embedding = nn.Linear(
            patch_len,
            d_model
        )

        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.n_patches,
                d_model
            )
        )

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=64,
            dropout=0.10,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=1
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        nn.init.trunc_normal_(
            self.pos_embedding,
            std=0.02
        )

    def forward(self, x):

        x = x.squeeze(-1)

        patches = x.unfold(
            dimension=1,
            size=self.patch_len,
            step=self.stride
        )

        z = self.embedding(
            patches
        )

        z = (
            z
            + self.pos_embedding
        )

        z = self.encoder(z)

        z = self.norm(z)

        # résumé global
        mean_pool = z.mean(dim=1)

        # dernier patch
        last_patch = z[:, -1, :]

        return torch.cat(
            [
                mean_pool,
                last_patch
            ],
            dim=1
        )

In [259]:
class MultiScaleResidualPatch(nn.Module):

    def __init__(
        self,
        seq_len,
        n_exog
    ):

        super().__init__()

        self.branch_short = PatchBranch(
            seq_len,
            patch_len=8,
            stride=4
        )

        self.branch_medium = PatchBranch(
            seq_len,
            patch_len=16,
            stride=8
        )

        self.branch_daily = PatchBranch(
            seq_len,
            patch_len=48,
            stride=24
        )

        self.exog_net = nn.Sequential(

            nn.Linear(
                n_exog,
                32
            ),

            nn.GELU(),

            nn.Dropout(0.05)
        )

        # 3 branches × 64 + 32 exog
        self.head = nn.Sequential(

            nn.Linear(
                224,
                64
            ),

            nn.GELU(),

            nn.Dropout(0.10),

            nn.Linear(
                64,
                1
            )
        )

    def forward(
        self,
        x_seq,
        x_exog
    ):

        short = self.branch_short(
            x_seq
        )

        medium = self.branch_medium(
            x_seq
        )

        daily = self.branch_daily(
            x_seq
        )

        exog = self.exog_net(
            x_exog
        )

        features = torch.cat(
            [
                short,
                medium,
                daily,
                exog
            ],
            dim=1
        )

        return self.head(
            features
        )

In [260]:
torch.manual_seed(42)

multi_patch_model = (
    MultiScaleResidualPatch(
        seq_len=336,
        n_exog=E_train.shape[1]
    )
    .to(device)
)

multi_params = sum(
    p.numel()
    for p in multi_patch_model.parameters()
    if p.requires_grad
)

print(
    "Multi-scale parameters :",
    multi_params
)

Multi-scale parameters : 47457


C:\Users\thiba\AppData\Local\Temp\ipykernel_17220\3562893126.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


In [261]:
multi_train_dataset = TensorDataset(
    torch.from_numpy(
        X_seq_train
    ),
    torch.from_numpy(
        E_train
    ),
    torch.from_numpy(
        y_seq_train_scaled[:, None]
    )
)

multi_val_dataset = TensorDataset(
    torch.from_numpy(
        X_seq_val
    ),
    torch.from_numpy(
        E_val
    ),
    torch.from_numpy(
        y_seq_val_scaled[:, None]
    )
)

multi_train_loader = DataLoader(
    multi_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

multi_val_loader = DataLoader(
    multi_val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [262]:
criterion_multi = nn.SmoothL1Loss()

optimizer_multi = torch.optim.AdamW(
    multi_patch_model.parameters(),
    lr=2e-4,
    weight_decay=5e-4
)

scaler_multi = torch.amp.GradScaler(
    "cuda"
)

MAX_SECONDS = 300
MAX_EPOCHS = 100
PATIENCE = 8

best_multi_rmse = np.inf
best_multi_epoch = None

patience_multi = 0
history_multi = []

MULTI_PATH = (
    "models/block6_multiscale_patch.pt"
)

start = time.time()

In [263]:
for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    multi_patch_model.train()

    total_loss = 0
    n_seen = 0
    timeout = False

    for xb, eb, yb in multi_train_loader:

        if (
            time.time() - start
            >= MAX_SECONDS
        ):

            timeout = True
            break

        xb = xb.to(
            device,
            non_blocking=True
        )

        eb = eb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        optimizer_multi.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = multi_patch_model(
                xb,
                eb
            )

            loss = criterion_multi(
                pred,
                yb
            )

        scaler_multi.scale(
            loss
        ).backward()

        scaler_multi.step(
            optimizer_multi
        )

        scaler_multi.update()

        total_loss += (
            loss.item()
            * len(xb)
        )

        n_seen += len(xb)

    if timeout:

        print(
            "Limite de 5 minutes atteinte."
        )

        break

    train_loss = (
        total_loss
        / n_seen
    )

    multi_patch_model.eval()

    preds = []

    with torch.no_grad():

        for xb, eb, _ in multi_val_loader:

            xb = xb.to(device)
            eb = eb.to(device)

            pred = multi_patch_model(
                xb,
                eb
            )

            preds.append(
                pred.cpu().numpy()
            )

    pred_val_scaled = (
        np.vstack(preds)
        .ravel()
    )

    pred_val_multi = (
        pred_val_scaled
        * seq_std
        + seq_mean
    )

    val_rmse = np.sqrt(
        mean_squared_error(
            y_seq_val,
            pred_val_multi
        )
    )

    history_multi.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_rmse": val_rmse
    })

    joblib.dump(
        history_multi,
        "models/block6_multiscale_history.joblib"
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss={train_loss:.5f} | "
        f"Val RMSE={val_rmse:.3f}"
    )

    if val_rmse < best_multi_rmse:

        best_multi_rmse = val_rmse
        best_multi_epoch = epoch

        patience_multi = 0

        torch.save(
            multi_patch_model.state_dict(),
            MULTI_PATH
        )

    else:

        patience_multi += 1

    if patience_multi >= PATIENCE:

        print(
            "Early stopping."
        )

        break

Epoch 001 | Loss=0.36788 | Val RMSE=205.350
Epoch 002 | Loss=0.36279 | Val RMSE=204.949
Epoch 003 | Loss=0.36082 | Val RMSE=204.384
Epoch 004 | Loss=0.35887 | Val RMSE=204.222
Epoch 005 | Loss=0.35589 | Val RMSE=204.328
Epoch 006 | Loss=0.35331 | Val RMSE=204.193
Epoch 007 | Loss=0.35200 | Val RMSE=203.611
Epoch 008 | Loss=0.34950 | Val RMSE=203.593
Epoch 009 | Loss=0.34658 | Val RMSE=203.439
Epoch 010 | Loss=0.34478 | Val RMSE=202.901
Epoch 011 | Loss=0.34198 | Val RMSE=203.196
Epoch 012 | Loss=0.33946 | Val RMSE=203.370
Epoch 013 | Loss=0.33643 | Val RMSE=202.932
Epoch 014 | Loss=0.33353 | Val RMSE=203.357
Epoch 015 | Loss=0.33003 | Val RMSE=202.935
Epoch 016 | Loss=0.32574 | Val RMSE=203.526
Epoch 017 | Loss=0.32341 | Val RMSE=203.583
Epoch 018 | Loss=0.32009 | Val RMSE=203.220
Early stopping.


In [264]:
multi_patch_model.load_state_dict(
    torch.load(
        MULTI_PATH,
        map_location=device,
        weights_only=True
    )
)

multi_patch_model.eval()

MultiScaleResidualPatch(
  (branch_short): PatchBranch(
    (embedding): Linear(in_features=8, out_features=32, bias=True)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
          )
          (linear1): Linear(in_features=32, out_features=64, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=64, out_features=32, bias=True)
          (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (branch_medium): PatchBranch(
    (embedding): Lin

In [265]:
multi_val_pred = []

with torch.no_grad():

    for xb, eb, _ in multi_val_loader:

        xb = xb.to(device)
        eb = eb.to(device)

        pred = multi_patch_model(
            xb,
            eb
        )

        multi_val_pred.append(
            pred.cpu().numpy()
        )

multi_val_pred = (
    np.vstack(
        multi_val_pred
    )
    .ravel()
)

multi_val_pred = (
    multi_val_pred
    * seq_std
    + seq_mean
)

In [266]:
eta_multi_results = []

for eta in np.linspace(
    0,
    1,
    11
):

    rmse = np.sqrt(
        mean_squared_error(
            y_seq_val,
            eta * multi_val_pred
        )
    )

    eta_multi_results.append([
        eta,
        rmse
    ])

eta_multi_df = pd.DataFrame(
    eta_multi_results,
    columns=[
        "eta",
        "Residual_RMSE"
    ]
)

print(
    eta_multi_df
)

best_eta_multi = (
    eta_multi_df.loc[
        eta_multi_df[
            "Residual_RMSE"
        ].idxmin(),
        "eta"
    ]
)

print(
    "Best multi-scale eta :",
    best_eta_multi
)

    eta  Residual_RMSE
0   0.0     205.531274
1   0.1     204.887469
2   0.2     204.326922
3   0.3     203.850322
4   0.4     203.458258
5   0.5     203.151219
6   0.6     202.929591
7   0.7     202.793655
8   0.8     202.743582
9   0.9     202.779437
10  1.0     202.901173
Best multi-scale eta : 0.8


In [267]:
multi_eval_dataset = TensorDataset(
    torch.from_numpy(
        X_seq_eval
    ),
    torch.from_numpy(
        E_eval
    )
)

multi_eval_loader = DataLoader(
    multi_eval_dataset,
    batch_size=256,
    shuffle=False
)

In [268]:
multi_pred = []

with torch.no_grad():

    for xb, eb in multi_eval_loader:

        xb = xb.to(device)
        eb = eb.to(device)

        pred = multi_patch_model(
            xb,
            eb
        )

        multi_pred.append(
            pred.cpu().numpy()
        )

multi_pred = (
    np.vstack(multi_pred)
    .ravel()
)

multi_pred = (
    multi_pred
    * seq_std
    + seq_mean
)

In [269]:
pred_multiscale_2025 = (
    base_stage1_gru
    +
    best_eta_multi
    * multi_pred
)

multiscale_metrics = regression_metrics(
    y_eval_gru,
    pred_multiscale_2025
)

print(
    "Multi-scale metrics :",
    multiscale_metrics
)

Multi-scale metrics : {'MSE': 32127.734023296038, 'RMSE': np.float64(179.24211007265018), 'MAE': 129.41929389909797, 'MAPE_%': 0.26120372110035084, 'R2': 0.9997151139277162}


In [270]:
compare_patch = pd.DataFrame([
    {
        "Model":
            "Patch Transformer",
        **patch_metrics
    },

    {
        "Model":
            "Multi-Scale Patch + Exog",
        **multiscale_metrics
    }
])

print(
    compare_patch.round(4)
)

                      Model         MSE      RMSE       MAE  MAPE_%      R2
0         Patch Transformer  31535.5289  177.5825  128.2453  0.2586  0.9997
1  Multi-Scale Patch + Exog  32127.7340  179.2421  129.4193  0.2612  0.9997


In [271]:
multiscale_gain = (
    100
    * (
        patch_metrics["RMSE"]
        -
        multiscale_metrics["RMSE"]
    )
    /
    patch_metrics["RMSE"]
)

print(
    "Gain vs Patch :",
    multiscale_gain,
    "%"
)

Gain vs Patch : -0.9345818251587782 %


In [272]:
joblib.dump(
    {
        "metrics":
            compare_patch,

        "best_eta":
            best_eta_multi,

        "best_epoch":
            best_multi_epoch,

        "parameters":
            multi_params,

        "history":
            history_multi,

        "gain_vs_patch":
            multiscale_gain
    },
    "models/block6_multiscale_patch_results.joblib"
)

['models/block6_multiscale_patch_results.joblib']

### Partie C — Analyse

Le Patch Transformer simple utilise une seule granularité temporelle.

Le modèle multi-échelle utilise simultanément des patches représentant environ :

$$
4h,\quad8h,\quad24h
$$

afin de capturer des structures temporelles de différentes durées.

Des variables météorologiques retardées et des variables calendaires cycliques sont également intégrées.

Le modèle combine donc :

$$
\text{structure temporelle}
+
\text{multi-échelle}
+
\text{contexte externe}
$$

Cette architecture est plus proche des approches récentes de short-term load forecasting qui combinent plusieurs échelles temporelles et des variables exogènes.

### Conclusion — Multi-Scale Patch Transformer

Le modèle multi-échelle avec variables exogènes n'améliore pas le Patch Transformer simple.

Les performances passent de :

$$
RMSE_{Patch}=177.58\text{ MW}
$$

à :

$$
RMSE_{MultiScale}=179.24\text{ MW}
$$

soit une dégradation d'environ :

$$
0.93\%
$$

Le MAE se dégrade également :

$$
128.25
\rightarrow
129.42\text{ MW}
$$

Le modèle multi-échelle apprend néanmoins une correction non nulle, avec :

$$
\eta=0.8
$$

mais la complexité supplémentaire n'apporte pas de meilleure généralisation.

Le Patch Transformer simple reste donc le meilleur modèle du Bloc 6.

## Performance Summary — Bloc 6

| Modèle | RMSE | MAE | MAPE | R² |
|---|---:|---:|---:|---:|
| Ridge + Weather | 209.66 | 151.71 | 0.3074 % | 0.9996 |
| Direct MLP | 288.52 | 222.92 | 0.4545 % | 0.9993 |
| Ridge + MLP1 | 182.22 | 132.22 | 0.2670 % | 0.9997 |
| Ridge + MLP1 + GRU | 181.81 | 131.87 | 0.2665 % | 0.9997 |
| Ridge + MLP1 + TCN | 180.94 | 131.26 | 0.2650 % | 0.9997 |
| Ridge + MLP1 + AR | 180.76 | 131.21 | 0.2651 % | 0.9997 |
| Ridge + MLP1 + MLP2 | 180.12 | 130.29 | 0.2629 % | 0.9997 |
| Ridge + MLP1 + Patch Transformer | **177.58** | **128.25** | **0.2586 %** | **0.99972** |
| Multi-Scale Patch + Exog | 179.24 | 129.42 | 0.2612 % | 0.99972 |

### Gain par rapport au Ridge

$$
\Delta RMSE \approx -15.30\%
$$

$$
\Delta MAE \approx -15.47\%
$$

$$
\Delta MSE \approx -28.26\%
$$

# Conclusion du Bloc 6

Le deep learning direct n'améliore pas la baseline linéaire.

En revanche, l'apprentissage résiduel permet d'exploiter la structure restante après le Ridge.

La première correction MLP réduit fortement l'erreur, puis les modèles temporels exploitent une structure résiduelle plus faible.

Parmi les modèles séquentiels, le Patch Transformer obtient la meilleure performance.

Le meilleur modèle du bloc est :

$$
\boxed{\text{Ridge + MLP1 + Patch Transformer}}
$$

avec :

$$
RMSE=177.58\text{ MW}
$$

contre :

$$
209.66\text{ MW}
$$

pour le Ridge initial.

Le gain total est d'environ :

$$
15.3\%
$$

sur le RMSE.

Les modèles plus complexes multi-échelles n'améliorent pas davantage la généralisation, ce qui montre que l'augmentation de complexité n'est pas systématiquement bénéfique.

In [273]:
# ============================================================
# EXPORT FINAL DU BLOC 6
# ============================================================

# ---------- Validation 2024 ----------

stage1_val_series = pd.Series(
    pred_stage1_2024,
    index=X_res_val_raw.index
)

pred_patch_2024 = (
    stage1_val_series.loc[
        val_seq_index
    ].to_numpy()
    + best_eta_patch * patch_val
)

y_patch_2024 = (
    y.loc[val_seq_index]
    .to_numpy()
)


# ---------- Evaluation 2025 ----------

pred_patch_2025 = np.asarray(
    pred_stage1_patch
)

y_patch_2025 = np.asarray(
    y_eval_gru
)


# ---------- Vérifications ----------

assert len(pred_patch_2024) == len(y_patch_2024)
assert len(pred_patch_2025) == len(y_patch_2025)

assert len(pred_patch_2025) == len(eval_seq_index)


# ---------- Etat final ----------

block6_final_state = {

    "val_index_2024":
        val_seq_index,

    "y_2024":
        y_patch_2024,

    "pred_patch_2024":
        pred_patch_2024,

    "eval_index_2025":
        eval_seq_index,

    "y_2025":
        y_patch_2025,

    "pred_patch_2025":
        pred_patch_2025,

    "best_eta_patch":
        best_eta_patch,

    "patch_metrics_2025":
        patch_metrics,

    "ridge_metrics_2025":
        ridge_metrics,

    "best_simple_model":
        "Ridge + MLP1 + Patch Transformer",

    "rmse_best":
        patch_metrics["RMSE"]
}

joblib.dump(
    block6_final_state,
    "models/block6_final_state.joblib"
)

print(
    "Bloc 6 sauvegardé."
)

print(
    "Best RMSE :",
    block6_final_state["rmse_best"]
)

Bloc 6 sauvegardé.
Best RMSE : 177.58245670759058


In [275]:
# ============================================================
# SAUVEGARDE STAGE 1 COMPLET — POUR BLOC 7
# ============================================================

joblib.dump(
    {
        "index_2024":
            X_res_val_raw.index,

        "y_2024":
            np.asarray(y_res_val),

        "pred_stage1_2024":
            np.asarray(pred_stage1_2024),

        "r2_2024":
            np.asarray(r2_2024),

        "index_2025":
            X_res_eval_raw.index,

        "y_2025":
            np.asarray(y_res_eval),

        "pred_stage1_2025":
            np.asarray(pred_residual_mlp),

        "r2_2025":
            np.asarray(r2_eval)
    },

    "models/block6_stage1_full_2024_2025.joblib"
)

print(
    "Stage 1 complet sauvegardé."
)

Stage 1 complet sauvegardé.
